In [ ]:
# ===========================================
# Colab：MNLogit（只用指定特征）+ 稳健版 HM IIA（修复QR），但是证明了拒绝IIA假设,证明MNLogit模型不是比较合适的模型
# ===========================================
!pip -q install pandas numpy scipy statsmodels tqdm openpyxl

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from collections import Counter
from tqdm.auto import tqdm
from scipy.stats import chi2
from scipy import linalg as la   # <-- 用 SciPy 的 QR（支持 pivoting=True）

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------ 配置 ------------
FILE_PATH = "/content/balanced_accidents_dataset (5) (3).csv"   # 改为你的路径
TARGET    = "grav"
OUT_DIR   = "/content/mnlogit_specified_stable"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_HM   = True
RARE_MIN = 150          # 稀有水平合并阈值（可再调）
TOPK_FOR_INTERACTION = {  # 交互仅取这些变量中频率最高的K个水平
    "secu": 3,
    "obs" : 3,
    "lum" : 3,
}

# 列名容错映射
def find_name(cands, cols):
    m = {c.lower(): c for c in cols}
    for k in cands:
        if k.lower() in m: return m[k.lower()]
    return None

REQUESTED = {
    "secu":    ["secu"],
    "obs":     ["obs","Obs"],
    "nbv":     ["nbv"],
    "age":     ["age"],
    "sexe":    ["sexe"],
    "traject": ["traject","trajet"],
    "lum":     ["lum","lighting","Lighting"],   # 仅用于交互
}

# ------------ 读取与映射 ------------
head = pd.read_csv(FILE_PATH, nrows=1)
all_cols = head.columns.tolist()
mapped = {k: find_name(v, all_cols) for k,v in REQUESTED.items()}
miss = [k for k,v in mapped.items() if v is None]
if miss:
    raise ValueError(f"以下必需列未找到：{miss}\n映射：{mapped}")

use_cols = [TARGET] + sorted(set(mapped.values()))
df = pd.read_csv(FILE_PATH, usecols=use_cols).copy()

# ------------ 工具函数 ------------
def merge_rare(s, min_count=RARE_MIN):
    s = s.astype(str)
    vc = s.value_counts(dropna=True)
    rare = vc[vc < min_count].index
    return s.where(~s.isin(rare), "__OTHER__")

def drop_zero_variance_and_make_fullrank(X, tol=1e-10):
    """丢零方差列 + 用 SciPy 的列主元QR选出满秩列"""
    keep = X.columns[X.std(ddof=0).to_numpy() > 0]
    X = X[keep]
    if X.shape[1] == 0:
        return X.iloc[:, []], []
    A = X.to_numpy(dtype=float, copy=False)
    # SciPy 的 QR，支持 pivoting
    Q, R, P = la.qr(A, mode="economic", pivoting=True)
    diag = np.abs(np.diag(R))
    if diag.size == 0:
        return X.iloc[:, []], []
    r = int(np.sum(diag > tol * diag[0]))  # 满秩列数
    keep_idx = P[:r]
    kept_cols = X.columns[keep_idx].tolist()
    return X.iloc[:, keep_idx], kept_cols

def build_X(df_in):
    """只用指定主效应 + 两类交互构建设计矩阵；返回X、y与列名细节"""
    df1 = df_in.copy()
    cat_main = [mapped[k] for k in ["secu","obs","nbv","sexe","traject"]]
    num_col  = mapped["age"]
    lum_col  = mapped["lum"]

    df1 = df1.dropna(subset=[TARGET] + cat_main + [num_col, lum_col]).copy()
    df1[TARGET] = df1[TARGET].astype(str)
    for c in cat_main + [lum_col]:
        df1[c] = merge_rare(df1[c], RARE_MIN)

    age = pd.to_numeric(df1[num_col], errors="coerce")
    age = (age - age.mean()) / (age.std(ddof=0) if age.std(ddof=0) != 0 else 1.0)
    df1[num_col] = age

    X_cat_main = pd.get_dummies(df1[cat_main], drop_first=True, dtype=float)

    X_secu = pd.get_dummies(df1[mapped["secu"]], drop_first=True, prefix=mapped["secu"], dtype=float)
    X_obs  = pd.get_dummies(df1[mapped["obs"]],  drop_first=True, prefix=mapped["obs"],  dtype=float)
    X_lum  = pd.get_dummies(df1[lum_col],        drop_first=True, prefix=lum_col,       dtype=float)

    def topk_cols(Xd, var_key):
        K = TOPK_FOR_INTERACTION.get(var_key, None)
        if not K or Xd.shape[1] <= K: return list(Xd.columns)
        freq = Xd.sum(axis=0).sort_values(ascending=False)
        return list(freq.index[:K])

    secu_cols = topk_cols(X_secu, "secu")
    obs_cols  = topk_cols(X_obs,  "obs")
    lum_cols  = topk_cols(X_lum,  "lum")

    # age × secu
    X_age_secu = pd.DataFrame(index=df1.index)
    for sc in secu_cols:
        X_age_secu[f"{num_col}__X__{sc}"] = df1[num_col].values * X_secu[sc].values

    # obs × lum
    X_obs_lum = pd.DataFrame(index=df1.index)
    for oc in obs_cols:
        for lc in lum_cols:
            X_obs_lum[f"{oc}__X__{lc}"] = X_obs[oc].values * X_lum[lc].values

    X = pd.concat([df1[[num_col]].astype(float), X_cat_main, X_age_secu, X_obs_lum], axis=1)
    X, kept = drop_zero_variance_and_make_fullrank(X)
    X = sm.add_constant(X, has_constant='add')
    y = df1[TARGET].copy()

    meta = {
        "num_col": num_col,
        "main_cols": list(X_cat_main.columns),
        "secu_inter_cols" : secu_cols,
        "obs_inter_cols"  : obs_cols,
        "lum_inter_cols"  : lum_cols,
        "kept_cols": [c for c in X.columns if c != "const"]
    }
    return X, y, meta

def relabel(y_series, baseline):
    alts = list(y_series.unique())
    other = [a for a in alts if a != baseline]
    labels = other + [baseline]
    mapping = {lab:i for i, lab in enumerate(labels)}
    return y_series.map(mapping).to_numpy(), labels

def fit_mnlogit(X, y_int):
    mod = sm.MNLogit(y_int, X)
    for method, iters in [("newton",300), ("bfgs",500), ("lbfgs",500)]:
        try:
            return mod.fit(method=method, maxiter=iters, disp=False), None
        except Exception as e:
            last_err = f"{method}: {e}"
    return None, last_err

def hm_iia_with_alignment(X_full, y_full, baseline):
    rows = []
    y_int_full, labels_full = relabel(y_full, baseline)
    res_full, err_full = fit_mnlogit(X_full, y_int_full)
    if res_full is None:
        rows.append({"baseline": baseline, "removed_alternative": None,
                     "chi2_stat": np.nan, "df": np.nan, "p_value": np.nan,
                     "error": f"FULL failed: {err_full}"})
        return pd.DataFrame(rows)

    nonbase_full = labels_full[:-1]
    full_cols    = list(X_full.columns)

    for rem in tqdm(nonbase_full, desc="IIA HM: removing alternative"):
        try:
            mask = (y_full != rem)
            Xr   = X_full.loc[mask].copy()
            yr   = y_full.loc[mask]

            # 删一类后再次做“零方差+满秩”
            Xr_noconst = Xr.drop(columns=["const"])
            Xr_clean, kept_r = drop_zero_variance_and_make_fullrank(Xr_noconst)
            Xr_clean = sm.add_constant(Xr_clean, has_constant='add')

            # 对齐列名
            common_cols = ["const"] + sorted(set(Xr_clean.columns) & set(full_cols) - {"const"})
            Xf_align = X_full[common_cols]
            Xr_align = Xr_clean[common_cols]

            # REDUCED 拟合
            y_int_r, labels_r = relabel(yr, baseline)
            res_r, err_r = fit_mnlogit(Xr_align, y_int_r)
            if res_r is None:
                rows.append({"baseline": baseline, "removed_alternative": rem,
                             "chi2_stat": np.nan, "df": np.nan, "p_value": np.nan,
                             "error": f"REDUCED failed: {err_r}"})
                continue

            nonbase_r = labels_r[:-1]
            keep_full_idx = [nonbase_full.index(l) for l in nonbase_r if l in nonbase_full]
            keep_r_idx    = [nonbase_r.index(l)    for l in nonbase_r if l in nonbase_full]

            def pack(res, kept_nonbase_idx):
                B = res.params.values
                b = B[:, kept_nonbase_idx].ravel(order='F')
                V = res.cov_params().values
                k = res.params.shape[0]
                idx = []
                for j in kept_nonbase_idx:
                    base = j*k
                    idx.extend(range(base, base+k))
                return b, V[np.ix_(idx, idx)]

            bF, VF = pack(res_full, keep_full_idx)
            bR, VR = pack(res_r,    keep_r_idx)

            diff  = bR - bF
            Vdiff = VR - VF
            Vinv  = np.linalg.pinv(Vdiff)
            stat  = float(diff.T @ Vinv @ diff)
            stat  = max(stat, 0.0) if np.isfinite(stat) else 0.0
            df_h  = diff.size
            pval  = 1.0 - chi2.cdf(stat, df_h)

            rows.append({"baseline": baseline, "removed_alternative": rem,
                         "chi2_stat": stat, "df": df_h, "p_value": pval})
        except Exception as ex:
            rows.append({"baseline": baseline, "removed_alternative": rem,
                         "chi2_stat": np.nan, "df": np.nan, "p_value": np.nan,
                         "error": str(ex)})
    return pd.DataFrame(rows)

# ------------ 构建设计矩阵（FULL）并拟合 ------------
X, y, meta = build_X(df)
alt_counts = Counter(y)
baseline = max(alt_counts.keys(), key=lambda a: alt_counts[a])

y_int, labels = relabel(y, baseline)
res, err = fit_mnlogit(X, y_int)
if res is None:
    raise RuntimeError(f"MNLogit FULL 拟合失败：{err}")

params = res.params.copy(); params.index.name = "feature"; params.reset_index(inplace=True)
params_csv = os.path.join(OUT_DIR, "MNLogit_params_specified_stable.csv")
params.to_csv(params_csv, index=False)

# ------------ HM IIA（稳健版）------------
if RUN_HM:
    print("开始 Hausman–McFadden IIA（稳健版） ...")
    hm_df = hm_iia_with_alignment(X, y, baseline)
    hm_csv = os.path.join(OUT_DIR, "HM_IIA_results_specified_stable.csv")
    hm_df.to_csv(hm_csv, index=False)

# ------------ 导出汇总 ------------
excel_path = os.path.join(OUT_DIR, "MNLogit_specified_stable.xlsx")
with pd.ExcelWriter(excel_path) as w:
    pd.DataFrame({"setting":["baseline","n_obs","n_features"],
                  "value":[baseline, len(y), X.shape[1]]}).to_excel(w, "summary", index=False)
    pd.DataFrame({"mapped_key":list(mapped.keys()),
                  "column_used":list(mapped.values())}).to_excel(w, "columns_mapping", index=False)
    pd.DataFrame({"kept_feature_cols": meta["kept_cols"]}).to_excel(w, "kept_cols_full", index=False)
    params.to_excel(w, "MNLogit_params", index=False)
    if RUN_HM: hm_df.to_excel(w, "HM_results", index=False)

print("\n===== 完成 =====")
print("baseline:", baseline)
print("参数表 CSV：", params_csv)
if RUN_HM: print("HM IIA 结果 CSV：", hm_csv)
print("汇总 Excel：", excel_path)


开始 Hausman–McFadden IIA（稳健版） ...


IIA HM: removing alternative:   0%|          | 0/3 [00:00<?, ?it/s]


===== 完成 =====
baseline: 2.0
参数表 CSV： /content/mnlogit_specified_stable/MNLogit_params_specified_stable.csv
HM IIA 结果 CSV： /content/mnlogit_specified_stable/HM_IIA_results_specified_stable.csv
汇总 Excel： /content/mnlogit_specified_stable/MNLogit_specified_stable.xlsx


/tmp/ipython-input-2518752883.py:252: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  "value":[baseline, len(y), X.shape[1]]}).to_excel(w, "summary", index=False)
/tmp/ipython-input-2518752883.py:254: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  "column_used":list(mapped.values())}).to_excel(w, "columns_mapping", index=False)
/tmp/ipython-input-2518752883.py:255: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"kept_feature_cols": meta["kept_cols"]}).to_excel(w, "kept_cols_full", index=False)
/tmp/ipython-input-2518752883.py:256: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  params.to_excel(w, "MNLogit_params", index=

In [ ]:
# ============================================================
# 方案A：Ordered Logit (+Brant) → 若违反 → Generalized Ordered Logit
# 评估：hold-out 测试集（Acc、macro-F1、混淆矩阵）
# 输出：系数表（含SE/95%CI）、边际效应、Brant检验、预测评估、Excel汇总
# 仅使用特征：secu, obs, plan, lum, sexe, trajet（哑变量, drop_first），age（z-score），age_secu（数值）
# 目标：grav_balanced（按 1<2<3 的有序）
# ============================================================

!pip -q install pandas numpy scipy statsmodels scikit-learn tqdm openpyxl

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm
from scipy.stats import norm, chi2

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH = "/content/balanced_accidents_dataset (5) (3).csv"  # ← 改成你的路径
TARGET    = "grav_balanced"
OUT_DIR   = "/content/ordered_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE = 0.2
RANDOM_STATE = 42
RARE_MIN  = 150  # 合并稀有水平，若仍不稳可调大

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str); vc = s.value_counts(dropna=True)
    rare = vc[vc<k].index
    return s.where(~s.isin(rare), "__OTHER__")

def prep_data(path):
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0 = pd.read_csv(path)
    miss = [c for c in need if c not in df0.columns]
    if miss and "age_secu" in miss: miss.remove("age_secu")
    if miss: raise ValueError(f"缺少列：{miss}")
    use = [c for c in need if c in df0.columns]
    df = df0[use].dropna().copy()
    # 目标转成 1<2<3 的有序整数
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)
    # 合并稀有水平
    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        df[c] = merge_rare(df[c], RARE_MIN)
    # 数值处理
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = (df["age"] - df["age"].mean())/(df["age"].std(ddof=0) if df["age"].std(ddof=0)!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)
    # one-hot
    X_cat = pd.get_dummies(df[["secu","obs","plan","lum","sexe","trajet"]], drop_first=True, dtype=float)
    X_num = df[["age"] + (["age_secu"] if "age_secu" in df.columns else [])].astype(float)
    X = pd.concat([X_num, X_cat], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]
    y = df[TARGET].copy()
    return X, y, df

def ci95_from_se(b, se):
    z = 1.96
    return (b - z*se, b + z*se)

def export_table(df, path):
    df.to_csv(path, index=False)
    return path

# ---------------- Brant 检验（近似实现） ----------------
def brant_test_like(X, y):
    """
    用 K-1 个阈值的累计logit：I(y > k) ~ X，比较各阈值的系数是否相等。
    返回：全局Wald检验 + 各变量的Wald检验（χ²、df、p）。
    """
    levels = np.sort(y.unique())
    thresholds = levels[:-1]  # k=1,...,K-1，对应 I(y>k)
    betas = []
    covs  = []
    for k in thresholds:
        yk = (y > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=200)
        b = res.params.drop("const")  # 只比斜率
        V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values)
        covs.append(V.values)
    betas = np.stack(betas, axis=0)            # shape: (K-1, p)
    p = betas.shape[1]; K1 = betas.shape[0]
    # 全局：所有阈值斜率相等 → β1=β2=...=βK-1
    # 构造差分向量 d = [β2-β1, β3-β1, ...] 长度 (K-2+1)*p = (K-1-1)*p
    vec = []
    Vblk = []
    for k in range(1, K1):
        vec.append(betas[k] - betas[0])
        # Cov(βk - β1) ≈ Cov(βk) + Cov(β1) （不同阈值估计近似独立）
        Vblk.append(covs[k] + covs[0])
    d = np.concatenate(vec)                        # ((K1-1)*p,)
    V = np.zeros(((K1-1)*p, (K1-1)*p))
    for i in range(K1-1):
        V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    chi2_stat = float(d.T @ Vinv @ d)
    df_w = d.size
    p_global = 1.0 - chi2.cdf(chi2_stat, df_w)

    # 按变量分别测：对每个 j，检验 β_{2j}=…=β_{K-1,j}
    var_rows=[]
    for j, name in enumerate(X.columns):
        vec_j = []
        Vj = np.zeros((K1-1, K1-1))
        for k in range(1, K1):
            vec_j.append(betas[k, j] - betas[0, j])
            Vj[k-1,k-1] = covs[k][j,j] + covs[0][j,j]
        dj = np.asarray(vec_j)
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj)
        dfj = dj.size
        pj  = 1.0 - chi2.cdf(statj, dfj)
        var_rows.append({"feature": name, "chi2_stat": statj, "df": dfj, "p_value": pj})
    global_row = pd.DataFrame([{"scope":"global", "chi2_stat": chi2_stat, "df": df_w, "p_value": p_global}])
    var_df = pd.DataFrame(var_rows)
    return global_row, var_df

# ---------------- 边际效应（AME，有限差分） ----------------
def average_marginal_effects_ordered(res_ord, X, y_levels, h=1e-4, n_boot=0, rng_seed=123):
    """
    对每个自变量 j，计算对 P(Y=r) 的平均边际效应（连续用微小扰动，虚拟变量用0→1变化），
    可选 bootstrap 置信区间（n_boot > 0）。
    """
    def probs(res, exog):
        # statsmodels OrderedModel 的 predict 给每类概率
        pr = res.model.predict(res.params, exog=exog)
        # 可能返回 DataFrame/ndarray，统一成 ndarray
        return np.asarray(pr)

    X_base = X.copy()
    pr0 = probs(res_ord, X_base)  # (n, R)
    R = pr0.shape[1]
    ame = []
    cols = list(X.columns)
    for j, name in enumerate(cols):
        x1 = X_base.copy()
        if set(np.unique(X_base[name])).issubset({0.0,1.0}):   # 哑变量：0->1
            x1[name] = 1.0
        else:                                                  # 连续：+h
            x1[name] = x1[name] + h
        pr1 = probs(res_ord, x1)
        dP = pr1 - pr0
        ame_j = dP.mean(axis=0)            # 对每一类的平均效应
        for r in range(R):
            ame.append({"feature":name, "class":int(y_levels[r]), "AME":float(ame_j[r])})

    ame_df = pd.DataFrame(ame)
    if n_boot>0:
        rng = np.random.default_rng(rng_seed)
        boot_rows=[]
        for b in tqdm(range(n_boot), desc="AME bootstrap"):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            pr0 = probs(res_ord, Xb)
            for j, name in enumerate(cols):
                x1 = Xb.copy()
                if set(np.unique(Xb[name])).issubset({0.0,1.0}): x1[name] = 1.0
                else: x1[name] = x1[name] + h
                pr1 = probs(res_ord, x1)
                dP = pr1 - pr0
                ame_j = dP.mean(axis=0)
                for r in range(R):
                    boot_rows.append({"feature":name, "class":int(y_levels[r]), "AME":float(ame_j[r]), "boot":b})
        boot_df = pd.DataFrame(boot_rows)
        # 百分位 CI
        ci_rows=[]
        for (f,c), g in boot_df.groupby(["feature","class"]):
            low, high = np.percentile(g["AME"], [2.5, 97.5])
            ci_rows.append({"feature":f,"class":c,"AME_ci_low":low,"AME_ci_high":high})
        ame_ci = pd.DataFrame(ci_rows)
        ame_df = ame_df.merge(ame_ci, on=["feature","class"], how="left")
    return ame_df

# ---------------- 主流程 ----------------
# 读取与拆分
X_all, y_all, df_all = prep_data(FILE_PATH)
classes = np.sort(y_all.unique())
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

# Ordered Logit（平行线）
ord_mod = OrderedModel(endog=y_train, exog=X_train, distr="logit")
ord_res = ord_mod.fit(method="bfgs", maxiter=1000, disp=False)

# Brant 检验
global_brant, var_brant = brant_test_like(X_train, y_train)
violated = (global_brant["p_value"].iloc[0] < 0.05)

# 评估 Ordered Logit
pred_proba = ord_mod.predict(ord_res.params, exog=X_test)
y_pred = np.asarray(pred_proba).argmax(axis=1) + classes.min()  # 转回类别编码
acc  = accuracy_score(y_test, y_pred)
mf1  = f1_score(y_test, y_pred, average="macro")
cm   = confusion_matrix(y_test, y_pred, labels=classes)

# 系数表（含SE/CI）
coef = ord_res.params
se   = ord_res.bse
rows=[]
for k,v in coef.items():
    if k.startswith("threshold"):  # cutpoints
        rows.append({"term":k,"coef":v,"se":se[k],"ci_low":v-1.96*se[k],"ci_high":v+1.96*se[k],"type":"threshold"})
    else:
        rows.append({"term":k,"coef":v,"se":se[k],"ci_low":v-1.96*se[k],"ci_high":v+1.96*se[k],"type":"slope"})
coef_df = pd.DataFrame(rows)

# 边际效应（可把 n_boot 调大拿CI；默认不Bootstrap以加快速度）
ame_df = average_marginal_effects_ordered(ord_res, X_train, classes, n_boot=0)

# 若 Brant 违反 → 广义有序 Logit（每个阈值一个累计logit）
gologit_tables = {}
if violated:
    g_rows=[]
    for k in classes[:-1]:
        yk = (y_train > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X_train, has_constant='add')).fit(disp=False, maxiter=1000)
        tmp = pd.DataFrame({
            "threshold": k,
            "term": res.params.index,
            "coef": res.params.values,
            "se": res.bse.values
        })
        tmp["ci_low"]  = tmp["coef"] - 1.96*tmp["se"]
        tmp["ci_high"] = tmp["coef"] + 1.96*tmp["se"]
        g_rows.append(tmp)
    gologit_tables["coefficients"] = pd.concat(g_rows, ignore_index=True)

# 预测评估（报告）
report_txt = classification_report(y_test, y_pred, digits=3)

# ---------------- 保存 ----------------
excel_path = os.path.join(OUT_DIR, "Ordered_and_Gologit_summary.xlsx")
with pd.ExcelWriter(excel_path) as w:
    pd.DataFrame({"metric":["accuracy","macro_f1"],"value":[acc, mf1]}).to_excel(w, "test_metrics", index=False)
    pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes]).to_excel(w, "confusion_matrix")
    coef_df.to_excel(w, "ordered_coefficients", index=False)
    ame_df.to_excel(w, "ordered_AME", index=False)
    global_brant.to_excel(w, "brant_global", index=False)
    var_brant.to_excel(w, "brant_by_feature", index=False)
    if violated:
        gologit_tables["coefficients"].to_excel(w, "gologit_coefficients", index=False)

# 同时导出 CSV
coef_csv  = os.path.join(OUT_DIR, "ordered_coefficients.csv")
ame_csv   = os.path.join(OUT_DIR, "ordered_AME.csv")
brant_g   = os.path.join(OUT_DIR, "brant_global.csv")
brant_v   = os.path.join(OUT_DIR, "brant_by_feature.csv")
pred_csv  = os.path.join(OUT_DIR, "test_predictions.csv")
pd.DataFrame({"y_test":y_test, "y_pred":y_pred}).to_csv(pred_csv, index=False)
coef_df.to_csv(coef_csv, index=False); ame_df.to_csv(ame_csv, index=False)
global_brant.to_csv(brant_g, index=False); var_brant.to_csv(brant_v, index=False)
if violated:
    gologit_csv = os.path.join(OUT_DIR, "gologit_coefficients.csv")
    gologit_tables["coefficients"].to_csv(gologit_csv, index=False)

# ---------------- 控制台摘要 ----------------
print("=== Ordered Logit 结果 ===")
print(f"测试集：Acc={acc:.3f}，Macro-F1={mf1:.3f}")
print("\nBrant 检验（global）：")
print(global_brant.to_string(index=False))
if (var_brant is not None):
    print("\nBrant 检验（按变量）：前10行")
    print(var_brant.head(10).to_string(index=False))
print("\n系数（前12行）：")
print(coef_df.head(12).to_string(index=False))
print("\n边际效应（前12行）：")
print(ame_df.head(12).to_string(index=False))
if violated:
    print("\n（已违反比例优势，已输出广义有序Logit各阈值系数到 Excel/CSV）")

print("\n分类报告：\n", report_txt)
print("\n结果已保存到：", excel_path)


/tmp/ipython-input-2216669053.py:240: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"metric":["accuracy","macro_f1"],"value":[acc, mf1]}).to_excel(w, "test_metrics", index=False)
/tmp/ipython-input-2216669053.py:241: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes]).to_excel(w, "confusion_matrix")
/tmp/ipython-input-2216669053.py:242: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  coef_df.to_excel(w, "ordered_coefficients", index=False)
/tmp/ipython-input-2216669053.py:243: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  ame_df.

=== Ordered Logit 结果 ===
测试集：Acc=0.614，Macro-F1=0.622

Brant 检验（global）：
 scope   chi2_stat  df  p_value
global 2797.381743  50      0.0

Brant 检验（按变量）：前10行
  feature    chi2_stat  df      p_value
      age 3.510898e+01   1 3.117604e-09
secu_11.0 1.091747e-07   1 9.997364e-01
secu_12.0 1.267123e-07   1 9.997160e-01
secu_13.0 1.109920e-07   1 9.997342e-01
 secu_2.0 1.243667e-07   1 9.997186e-01
secu_21.0 1.443334e-07   1 9.996969e-01
secu_22.0 1.377183e-07   1 9.997039e-01
secu_23.0 1.409066e-07   1 9.997005e-01
 secu_3.0 1.314549e-07   1 9.997107e-01
secu_31.0 1.269791e-07   1 9.997157e-01

系数（前12行）：
     term      coef       se    ci_low   ci_high  type
      age  0.134232 0.013858  0.107071  0.161393 slope
secu_11.0 -4.368982 0.716481 -5.773285 -2.964679 slope
secu_12.0 -3.380106 0.725937 -4.802943 -1.957270 slope
secu_13.0 -4.249847 0.716998 -5.655164 -2.844530 slope
 secu_2.0  0.200247 1.010539 -1.780410  2.180904 slope
secu_21.0 -4.833784 0.717516 -6.240116 -3.427452 slope
secu_22

In [ ]:
# ============================================================
# Generalized Ordered Logit (Partial Proportional Odds) – Reviewer-ready
# A) 统计前提：Brant检验；约束(平行) vs 放松(部分比例优势) 的 LR/AIC/BIC/伪R²；阈值单调；阈值分层系数&95%CI
# B) 外推泛化：Accuracy / Macro-F1 / Quadratic κ / Ordinal MAE / Multiclass Brier + 混淆矩阵
# 变量：secu, obs, plan, lum, sexe, trajet（哑变量drop_first），age（z-score），age_secu（数值）
# 目标：grav_balanced（1<2<3）
# ============================================================

!pip -q install pandas numpy scipy statsmodels scikit-learn tqdm openpyxl lxml html5lib

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
from tqdm.auto import tqdm
from scipy.special import expit
from scipy.stats import chi2
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH = "/content/balanced_remapping_data (1).csv"   # ← 改成你的CSV路径
TARGET    = "grav_balanced"
OUT_DIR   = "/content/gologit_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE = 0.2
RANDOM_STATE = 42
RARE_MIN  = 150           # 合并稀有水平（如不稳可调大）
ALPHA_VAR = 0.05          # 逐变量非平行显著性阈值
FORCE_FREE = ["age"]      # 至少放松这些变量（与 Brant 结合，常见是 age）
N_BOOT_AME = 0            # 边际效应bootstrap次数（0=不做；建议 200/500）

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str); vc = s.value_counts(dropna=True); return s.where(~s.isin(vc[vc<k].index), "__OTHER__")

def prep_data(path):
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0  = pd.read_csv(path)
    use  = [c for c in need if c in df0.columns]
    if "grav_balanced" not in use: raise ValueError("缺少 grav_balanced")
    df = df0[use].dropna().copy()
    # 目标整数 1/2/3
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)
    # 合并稀有水平
    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        df[c] = merge_rare(df[c], RARE_MIN)
    # 数值
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = (df["age"] - df["age"].mean())/(df["age"].std(ddof=0) if df["age"].std(ddof=0)!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)
    # one-hot
    X_cat = pd.get_dummies(df[["secu","obs","plan","lum","sexe","trajet"]], drop_first=True, dtype=float)
    X_num = df[["age"] + (["age_secu"] if "age_secu" in df.columns else [])].astype(float)
    X = pd.concat([X_num, X_cat], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]   # 去零方差列
    y = df[TARGET].copy()
    return X, y, df

def brant_test_like(X, y):
    """
    K-1 个累计logit：I(y>k)~X，检验不同阈值的系数是否相等。
    返回：全局Wald + 逐变量Wald（χ²、df、p）。
    """
    levels = np.sort(y.unique()); th = levels[:-1]
    betas, covs = [], []
    for k in th:
        yk = (y > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=400)
        b = res.params.drop("const"); V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values); covs.append(V.values)
    betas = np.stack(betas, axis=0)  # (K-1, p)
    p = betas.shape[1]; K1 = betas.shape[0]

    # 全局：β2-β1, β3-β1, ...
    diffs, Vblk = [], []
    for k in range(1, K1):
        diffs.append(betas[k]-betas[0])
        Vblk.append(covs[k]+covs[0])
    d = np.concatenate(diffs); V = np.zeros(((K1-1)*p, (K1-1)*p))
    for i in range(K1-1): V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    chi2_stat = float(d.T @ Vinv @ d); df_w = d.size; p_global = 1.0 - chi2.cdf(chi2_stat, df_w)
    global_row = pd.DataFrame([{"scope":"global","chi2_stat":chi2_stat,"df":df_w,"p_value":p_global}])

    # 逐变量
    rows=[]
    for j,name in enumerate(X.columns):
        dj = np.array([betas[k,j]-betas[0,j] for k in range(1,K1)])
        Vj = np.diag([covs[k][j,j]+covs[0][j,j] for k in range(1,K1)])
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj); dfj = dj.size; pj = 1.0 - chi2.cdf(statj, dfj)
        rows.append({"feature":name,"chi2_stat":statj,"df":dfj,"p_value":pj})
    return global_row, pd.DataFrame(rows)

def stack_threshold_data(X, y):
    """
    叠栈数据：对每个样本与每个阈值 k=1..K-1，构造一行：yk=I(y>k)。
    返回 long_df（含 obs_id, thr, y_bin）以及阈值dummy矩阵。
    """
    classes = np.sort(y.unique()); th = classes[:-1]
    n = len(y); rep = len(th)
    long = pd.DataFrame({
        "obs_id": np.repeat(np.arange(n), rep),
        "thr":    np.tile(th, n),
        "y_bin":  (np.repeat(y.to_numpy(), rep) > np.tile(th, n)).astype(int),
    })
    Xrep = np.repeat(X.to_numpy(), rep, axis=0)
    long = pd.concat([long, pd.DataFrame(Xrep, columns=X.columns)], axis=1)
    thr_dum = pd.get_dummies(long["thr"].astype(str), drop_first=False, dtype=int) # 所有阈值dummy当作阈值截距
    thr_dum.columns = [f"thr_{c}" for c in thr_dum.columns]
    return long.reset_index(drop=True), thr_dum

def fit_stacked_logit(long_df, thr_dum, X, free_vars):
    """
    叠栈logit：阈值截距=thr_dum；对free_vars添加阈值交互，其余变量保持平行。
    无显式常数项（否则与 thr_dum共线）。
    """
    Z = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    # 交互：以第一个阈值为reference，只对其余阈值添加差异项（使可识别）
    thr_cols = [c for c in thr_dum.columns]
    ref_thr = thr_cols[0]
    for v in free_vars:
        for t in thr_cols[1:]:
            Z[f"{v}__x__{t}"] = long_df[v].to_numpy() * thr_dum[t].to_numpy()
    # 拟合（cluster-robust by obs_id）
    res = sm.Logit(long_df["y_bin"], Z).fit(disp=False, maxiter=400, cov_type="cluster",
                                            cov_kwds={"groups": long_df["obs_id"]})
    return res, Z, ref_thr, thr_cols

def refit_with_constraints(long_df, thr_dum, X):
    """
    约束模型（全部平行）：不含任何交互，只保留阈值截距 + 共用斜率。
    """
    Zc = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    resC = sm.Logit(long_df["y_bin"], Zc).fit(disp=False, maxiter=400, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resC, Zc

def refit_null(long_df, thr_dum):
    """
    空模型：仅阈值截距，无自变量（用于伪R²）。
    """
    Zn = thr_dum.copy()
    resN = sm.Logit(long_df["y_bin"], Zn).fit(disp=False, maxiter=200, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resN, Zn

def lr_test(res_full, res_small):
    LR = 2*(res_full.llf - res_small.llf)
    df = res_full.df_model - res_small.df_model
    p = 1.0 - chi2.cdf(LR, df if df>0 else 1)
    return LR, int(df), float(p)

def extract_threshold_coefs(res, Xcols, thr_cols, free_vars, ref_thr):
    """
    输出每个阈值下的系数（含OR和95%CI）：
    平行变量：同一套系数；
    放松变量：ref_thr 用 β，其他阈值用 β + Δβ(t)。
    """
    coefs = res.params; cov = res.cov_params()
    rows=[]
    for t in thr_cols:
        # 阈值截距
        b0 = coefs[t]; se0 = np.sqrt(cov.loc[t,t]); rows.append(
            {"threshold":t, "term":"intercept", "coef":b0, "se":se0,
             "ci_low":b0-1.96*se0, "ci_high":b0+1.96*se0, "OR":np.nan}
        )
        for v in Xcols:
            if v in free_vars:
                if t==ref_thr:
                    b = coefs.get(v, 0.0); var = cov.loc[v,v]
                else:
                    dv = f"{v}__x__{t}"
                    b = coefs.get(v, 0.0) + coefs.get(dv, 0.0)
                    # Var(β+Δ)=Var(β)+Var(Δ)+2Cov(β,Δ)
                    var = cov.loc[v,v] + cov.loc[dv,dv] + 2*cov.loc[v,dv]
            else:
                b = coefs.get(v, 0.0); var = cov.loc[v,v]
            se = np.sqrt(var)
            rows.append({"threshold":t, "term":v, "coef":b, "se":se,
                         "ci_low":b-1.96*se, "ci_high":b+1.96*se, "OR":np.exp(b)})
    return pd.DataFrame(rows)

def predict_gologit(res, X, thr_cols, free_vars, ref_thr):
    """
    用叠栈logit系数恢复累积概率 P(Y>k)，再转成类别概率 p1,p2,p3。
    """
    coefs = res.params
    # 累积阈值：按 thr_cols 顺序计算 P(Y>k)
    Pgt = []
    for t in thr_cols:
        eta = coefs[t]  # 阈值截距
        # 平行项
        for v in X.columns:
            if v not in free_vars:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
        # 放松项
        for v in free_vars:
            if t == ref_thr:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
            else:
                eta += X[v].to_numpy() * (coefs.get(v, 0.0) + coefs.get(f"{v}__x__{t}", 0.0))
        Pgt.append(expit(eta))
    Pgt = np.vstack(Pgt)                # shape: (K-1, n)
    # 类别概率（K=3）：p1=1-P(>1), p2=P(>1)-P(>2), p3=P(>2)
    p1 = 1.0 - Pgt[0]; p3 = Pgt[-1]; p2 = Pgt[0] - Pgt[-1]
    P = np.vstack([p1, p2, p3]).T
    # 归一保障
    P = np.clip(P, 1e-12, 1-1e-12); P = P / P.sum(axis=1, keepdims=True)
    return P

def ordered_metrics(y_true, y_pred, prob):
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro")
    kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = np.mean(np.abs(y_true - y_pred))
    # Multiclass Brier（平均每类平方误差之和 / K）
    K = prob.shape[1]
    Y = np.eye(K)[(y_true-1)]  # 1..K → 0..K-1
    brier = np.mean(np.sum((prob - Y)**2, axis=1)) / K
    return acc, mf1, kappa, mae, brier

def average_marginal_effects_gologit(res, X, thr_cols, free_vars, ref_thr, h=1e-4, n_boot=0, seed=123):
    """
    AME：对每个自变量 j，对每一类概率的平均影响（连续 +h；哑变量 0→1）。
    可选 bootstrap 95%CI。
    """
    rng = np.random.default_rng(seed)
    def probs(ex):
        return predict_gologit(res, ex, thr_cols, free_vars, ref_thr)
    baseP = probs(X); K = baseP.shape[1]
    out=[]
    for v in X.columns:
        X1 = X.copy()
        if set(np.unique(X[v])).issubset({0.0,1.0}): X1[v] = 1.0
        else: X1[v] = X1[v] + h
        dP = probs(X1) - baseP
        ame = dP.mean(axis=0)
        for k in range(K):
            out.append({"feature":v, "class":k+1, "AME":float(ame[k])})
    ame_df = pd.DataFrame(out)

    if n_boot>0:
        boots=[]
        for b in tqdm(range(n_boot), desc="AME bootstrap"):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            Pb = probs(Xb)
            for v in X.columns:
                X1 = Xb.copy()
                if set(np.unique(Xb[v])).issubset({0.0,1.0}): X1[v] = 1.0
                else: X1[v] = X1[v] + h
                dP = probs(X1) - Pb
                ame = dP.mean(axis=0)
                for k in range(K):
                    boots.append({"feature":v,"class":k+1,"AME":float(ame[k]),"boot":b})
        boots = pd.DataFrame(boots)
        cis=[]
        for (f,c), g in boots.groupby(["feature","class"]):
            lo, hi = np.percentile(g["AME"], [2.5, 97.5])
            cis.append({"feature":f,"class":c,"AME_ci_low":lo,"AME_ci_high":hi})
        ame_df = ame_df.merge(pd.DataFrame(cis), on=["feature","class"], how="left")
    return ame_df

# ---------------- 1) 读取 & Ordered Logit & Brant ----------------
X_all, y_all, df_all = prep_data(FILE_PATH)
classes = np.sort(y_all.unique())

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

po = OrderedModel(endog=y_tr, exog=X_tr, distr="logit")
po_res = po.fit(method="bfgs", maxiter=1000, disp=False)

# Brant（在训练集上）
brant_global, brant_by = brant_test_like(X_tr, y_tr)

# 确定要放松的变量（逐变量 p<ALPHA 或列在 FORCE_FREE）
violators = set(brant_by.loc[brant_by["p_value"]<ALPHA_VAR, "feature"].tolist())
for v in FORCE_FREE:
    if v in X_tr.columns: violators.add(v)
free_vars = sorted(list(violators))
if len(free_vars)==0:
    free_vars = ["age"] if "age" in X_tr.columns else []
print("将放松非平行的变量：", free_vars)

# ---------------- 2) 叠栈logit：约束 vs 放松（部分比例优势） ----------------
long_tr, thr_dum_tr = stack_threshold_data(X_tr, y_tr)

# 约束（全部平行）
res_con, Zc = refit_with_constraints(long_tr, thr_dum_tr, X_tr)

# 放松（只对 free_vars）
res_free, Zf, ref_thr, thr_cols = fit_stacked_logit(long_tr, thr_dum_tr, X_tr, free_vars)

# 空模型（仅阈值截距）
res_null, Zn = refit_null(long_tr, thr_dum_tr)

# 似然比 / AIC / BIC / 伪R²
LR, df_lr, p_lr = lr_test(res_free, res_con)
fit_table = pd.DataFrame([
    {"model":"Constrained_PO(stacked)","llf":res_con.llf,"df_model":res_con.df_model,"aic":res_con.aic,"bic":res_con.bic,
     "pseudoR2_vsNull": 1 - (res_con.llf / res_null.llf)},
    {"model":"Partial_PO(Free vars)","llf":res_free.llf,"df_model":res_free.df_model,"aic":res_free.aic,"bic":res_free.bic,
     "pseudoR2_vsNull": 1 - (res_free.llf / res_null.llf)},
])
lr_row = pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}])

# 阈值分层系数表（含OR与95%CI）
coef_by_thr = extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)

# cutpoints 单调性（截距应随阈值递增）
thr_order = coef_by_thr.query("term=='intercept'")[["threshold","coef"]]
thr_monotone = np.all(np.diff(thr_order["coef"].values) > 0)  # TRUE=单调

# ---------------- 3) 测试集预测 + 指标 ----------------
# 用放松模型作为最终模型
# a) 先在训练集的阈值结构上拟合得到 thr_cols/ref_thr（与训练一致）
P_te = predict_gologit(res_free, X_te, thr_cols, free_vars, ref_thr)
y_pred = P_te.argmax(axis=1) + classes.min()

acc, mf1, kappa, mae, brier = ordered_metrics(y_te.to_numpy(), y_pred, P_te)
cm = confusion_matrix(y_te, y_pred, labels=classes)
report_txt = classification_report(y_te, y_pred, digits=3)

# ---------------- 4) 边际效应（可选bootstrap 95%CI） ----------------
ame_df = average_marginal_effects_gologit(res_free, X_tr, thr_cols, free_vars, ref_thr, n_boot=N_BOOT_AME)

# ---------------- 5) 导出 Excel ----------------
excel = os.path.join(OUT_DIR, "Gologit_review_ready.xlsx")
with pd.ExcelWriter(excel) as w:
    # A) 统计前提
    pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
    brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
    fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
    lr_row.to_excel(w, "A3_LR_test_partial_vs_constrained", index=False)
    coef_by_thr.to_excel(w, "A4_threshold_level_coefs", index=False)
    pd.DataFrame({"threshold_order":thr_order["threshold"],"intercept":thr_order["coef"],"is_monotone":[thr_monotone]*len(thr_order)}).to_excel(w,"A5_cutpoint_monotonicity",index=False)
    # B) 外推指标
    pd.DataFrame({"metric":["Accuracy","MacroF1","QuadraticKappa","OrdinalMAE","MulticlassBrier"],
                  "value":[acc,mf1,kappa,mae,brier]}).to_excel(w,"B1_test_metrics",index=False)
    pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes]).to_excel(w,"B2_confusion_matrix")
    pd.DataFrame({"obs_id":np.arange(len(y_te)),"y_test":y_te,"y_pred":y_pred}).to_excel(w,"B3_test_predictions",index=False)
    ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)

# ---------------- 6) 控制台摘要（给论文/回复可直接引用） ----------------
print("=== 统计前提（结构） ===")
print("Brant 全局：", brant_global.to_string(index=False))
print("\n逐变量（p值最小前10）：\n", brant_by.sort_values("p_value").head(10).to_string(index=False))
print("\n将放松的变量：", free_vars)
print("\n约束 vs 放松 的拟合：\n", fit_table.to_string(index=False))
print("\n似然比检验（放松 vs 约束）：", lr_row.to_string(index=False))
print("\n阈值截距是否单调递增？ ->", thr_monotone)

print("\n=== 外推（测试集） ===")
print(f"Accuracy={acc:.3f}, Macro-F1={mf1:.3f}, Quadratic κ={kappa:.3f}, Ordinal MAE={mae:.3f}, Brier={brier:.4f}")
print("混淆矩阵：\n", cm)
print("\n分类报告：\n", report_txt)

print("\nExcel 文件：", excel)


将放松非平行的变量： ['age', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_10.0', 'obs_11.0', 'obs_12.0', 'obs_13.0', 'obs_14.0', 'obs_15.0', 'obs_2.0', 'obs_3.0', 'obs_4.0', 'obs_5.0', 'obs_6.0', 'obs_8.0', 'obs_9.0', 'plan_1', 'secu_1', 'secu_3', 'sexe_2', 'trajet_1.0', 'trajet_2.0', 'trajet_3.0']


/tmp/ipython-input-2336365098.py:335: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
/tmp/ipython-input-2336365098.py:336: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
/tmp/ipython-input-2336365098.py:337: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
/tmp/ipython-input-2336365098.py:338: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the a

=== 统计前提（结构） ===
Brant 全局：  scope   chi2_stat  df  p_value
global 1696.898505  36      0.0

逐变量（p值最小前10）：
  feature  chi2_stat  df      p_value
  secu_1 399.184551   1 0.000000e+00
 obs_1.0  84.412283   1 0.000000e+00
  sexe_2 170.067583   1 0.000000e+00
 obs_8.0 125.760226   1 0.000000e+00
     age  68.471715   1 1.110223e-16
   lum_5  62.887657   1 2.220446e-15
  plan_1  41.588883   1 1.126319e-10
 obs_6.0  41.224171   1 1.357336e-10
obs_15.0  38.932018   1 4.388238e-10
obs_14.0  37.735035   1 8.103598e-10

将放松的变量： ['age', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_10.0', 'obs_11.0', 'obs_12.0', 'obs_13.0', 'obs_14.0', 'obs_15.0', 'obs_2.0', 'obs_3.0', 'obs_4.0', 'obs_5.0', 'obs_6.0', 'obs_8.0', 'obs_9.0', 'plan_1', 'secu_1', 'secu_3', 'sexe_2', 'trajet_1.0', 'trajet_2.0', 'trajet_3.0']

约束 vs 放松 的拟合：
                   model           llf  df_model          aic          bic  pseudoR2_vsNull
Constrained_PO(stacked) -17734.124938      37.0 35544.249876 35873.581706         0.

In [ ]:
# ============================================================
# Generalized Ordered Logit (Partial Proportional Odds) – Reviewer-ready
# 支持自定义有序顺序（如 2<1<3），训练用“等级化标签”，对外显示可反映射回原编码
# ============================================================

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
from scipy.special import expit
from scipy.stats import chi2
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH   = "//content/crash_data(final).csv"   # ← 改成你的CSV路径
TARGET      = "grav_balanced"
OUT_DIR     = "/content/gologit_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE   = 0.2
RANDOM_STATE= 42
RARE_MIN    = 150
ALPHA_VAR   = 0.05
FORCE_FREE  = ["age"]
N_BOOT_AME  = 0

### NEW: 业务顺序（从低到高的严重度）。你的场景是 2(无伤)<1(轻伤)<3(重伤)：
ORDINAL_ORDER = [2, 1, 3]   # 如已是自然递增(1<2<3)可设为 None 或 [1,2,3]

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str); vc = s.value_counts(dropna=True); return s.where(~s.isin(vc[vc<k].index), "__OTHER__")

def _build_rank_maps(y_series, ordinal_order):
    """
    根据业务顺序生成 rank_map 与 inv_rank_map:
      原标签 -> 等级(1..K)； 等级 -> 原标签
    """
    if ordinal_order is None:
        # 若未指定，则按出现的唯一值从小到大做自然排序
        ordered = sorted(pd.unique(y_series))
    else:
        ordered = list(ordinal_order)
    rank_map = {v: i+1 for i, v in enumerate(ordered)}
    inv_rank_map = {r: v for v, r in rank_map.items()}
    return rank_map, inv_rank_map, ordered

def prep_data(path, ordinal_order=None):
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0  = pd.read_csv(path)
    use  = [c for c in need if c in df0.columns]
    if "grav_balanced" not in use: raise ValueError("缺少 grav_balanced")
    df = df0[use].dropna().copy()

    # 目标为整数
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

    ### NEW: 建立映射并把 y 转为真正 1<2<3 的等级
    rank_map, inv_rank_map, ordered_original = _build_rank_maps(df[TARGET], ordinal_order)
    y_orig = df[TARGET].copy()                                 # 备份原标签
    y_rank = df[TARGET].map(rank_map).astype(int)              # 等级化标签(1..K)

    # 合并稀有水平
    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        df[c] = merge_rare(df[c], RARE_MIN)

    # 数值
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = (df["age"] - df["age"].mean())/(df["age"].std(ddof=0) if df["age"].std(ddof=0)!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)

    # one-hot
    X_cat = pd.get_dummies(df[["secu","obs","plan","lum","sexe","trajet"]], drop_first=True, dtype=float)
    X_num = df[["age"] + (["age_secu"] if "age_secu" in df.columns else [])].astype(float)
    X = pd.concat([X_num, X_cat], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]   # 去零方差列

    return X, y_rank, y_orig, df, rank_map, inv_rank_map, ordered_original

def brant_test_like(X, y_rank):
    levels = np.sort(y_rank.unique()); th = levels[:-1]
    betas, covs = [], []
    for k in th:
        yk = (y_rank > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=400)
        b = res.params.drop("const"); V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values); covs.append(V.values)
    betas = np.stack(betas, axis=0)  # (K-1, p)
    p = betas.shape[1]; K1 = betas.shape[0]

    diffs, Vblk = [], []
    for k in range(1, K1):
        diffs.append(betas[k]-betas[0])
        Vblk.append(covs[k]+covs[0])
    d = np.concatenate(diffs); V = np.zeros(((K1-1)*p, (K1-1)*p))
    for i in range(K1-1): V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    chi2_stat = float(d.T @ Vinv @ d); df_w = d.size; p_global = 1.0 - chi2.cdf(chi2_stat, df_w)
    global_row = pd.DataFrame([{"scope":"global","chi2_stat":chi2_stat,"df":df_w,"p_value":p_global}])

    rows=[]
    for j,name in enumerate(X.columns):
        dj = np.array([betas[k,j]-betas[0,j] for k in range(1,K1)])
        Vj = np.diag([covs[k][j,j]+covs[0][j,j] for k in range(1,K1)])
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj); dfj = dj.size; pj = 1.0 - chi2.cdf(statj, dfj)
        rows.append({"feature":name,"chi2_stat":statj,"df":dfj,"p_value":pj})
    return global_row, pd.DataFrame(rows)

def stack_threshold_data(X, y_rank):
    classes = np.sort(y_rank.unique()); th = classes[:-1]
    n = len(y_rank); rep = len(th)
    long = pd.DataFrame({
        "obs_id": np.repeat(np.arange(n), rep),
        "thr":    np.tile(th, n),
        "y_bin":  (np.repeat(y_rank.to_numpy(), rep) > np.tile(th, n)).astype(int),
    })
    Xrep = np.repeat(X.to_numpy(), rep, axis=0)
    long = pd.concat([long, pd.DataFrame(Xrep, columns=X.columns)], axis=1)
    thr_dum = pd.get_dummies(long["thr"].astype(str), drop_first=False, dtype=int)
    thr_dum.columns = [f"thr_{c}" for c in thr_dum.columns]
    return long.reset_index(drop=True), thr_dum

def fit_stacked_logit(long_df, thr_dum, X, free_vars):
    Z = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    thr_cols = [c for c in thr_dum.columns]
    ref_thr = thr_cols[0]
    for v in free_vars:
        for t in thr_cols[1:]:
            Z[f"{v}__x__{t}"] = long_df[v].to_numpy() * thr_dum[t].to_numpy()
    res = sm.Logit(long_df["y_bin"], Z).fit(disp=False, maxiter=400, cov_type="cluster",
                                            cov_kwds={"groups": long_df["obs_id"]})
    return res, Z, ref_thr, thr_cols

def refit_with_constraints(long_df, thr_dum, X):
    Zc = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    resC = sm.Logit(long_df["y_bin"], Zc).fit(disp=False, maxiter=400, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resC, Zc

def refit_null(long_df, thr_dum):
    Zn = thr_dum.copy()
    resN = sm.Logit(long_df["y_bin"], Zn).fit(disp=False, maxiter=200, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resN, Zn

def lr_test(res_full, res_small):
    LR = 2*(res_full.llf - res_small.llf)
    df = res_full.df_model - res_small.df_model
    p = 1.0 - chi2.cdf(LR, df if df>0 else 1)
    return LR, int(df), float(p)

def extract_threshold_coefs(res, Xcols, thr_cols, free_vars, ref_thr):
    coefs = res.params; cov = res.cov_params()
    rows=[]
    for t in thr_cols:
        b0 = coefs[t]; se0 = np.sqrt(cov.loc[t,t]); rows.append(
            {"threshold":t, "term":"intercept", "coef":b0, "se":se0,
             "ci_low":b0-1.96*se0, "ci_high":b0+1.96*se0, "OR":np.nan}
        )
        for v in Xcols:
            if v in free_vars:
                if t==ref_thr:
                    b = coefs.get(v, 0.0); var = cov.loc[v,v]
                else:
                    dv = f"{v}__x__{t}"
                    b = coefs.get(v, 0.0) + coefs.get(dv, 0.0)
                    var = cov.loc[v,v] + cov.loc[dv,dv] + 2*cov.loc[v,dv]
            else:
                b = coefs.get(v, 0.0); var = cov.loc[v,v]
            se = np.sqrt(var)
            rows.append({"threshold":t, "term":v, "coef":b, "se":se,
                         "ci_low":b-1.96*se, "ci_high":b+1.96*se, "OR":np.exp(b)})
    return pd.DataFrame(rows)

def predict_gologit(res, X, thr_cols, free_vars, ref_thr):
    coefs = res.params
    Pgt = []
    for t in thr_cols:
        eta = coefs[t]
        for v in X.columns:
            if v not in free_vars:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
        for v in free_vars:
            if t == ref_thr:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
            else:
                eta += X[v].to_numpy() * (coefs.get(v, 0.0) + coefs.get(f"{v}__x__{t}", 0.0))
        Pgt.append(expit(eta))
    Pgt = np.vstack(Pgt)                # (K-1, n)
    p1 = 1.0 - Pgt[0]; p3 = Pgt[-1]; p2 = Pgt[0] - Pgt[-1]
    P = np.vstack([p1, p2, p3]).T
    P = np.clip(P, 1e-12, 1-1e-12); P = P / P.sum(axis=1, keepdims=True)
    return P

def ordered_metrics(y_true_rank, y_pred_rank, prob):
    acc = accuracy_score(y_true_rank, y_pred_rank)
    mf1 = f1_score(y_true_rank, y_pred_rank, average="macro")
    kappa = cohen_kappa_score(y_true_rank, y_pred_rank, weights="quadratic")
    mae = np.mean(np.abs(y_true_rank - y_pred_rank))
    K = prob.shape[1]
    Y = np.eye(K)[(y_true_rank-1)]
    brier = np.mean(np.sum((prob - Y)**2, axis=1)) / K
    return acc, mf1, kappa, mae, brier

def average_marginal_effects_gologit(res, X, thr_cols, free_vars, ref_thr, h=1e-4, n_boot=0, seed=123):
    rng = np.random.default_rng(seed)
    def probs(ex): return predict_gologit(res, ex, thr_cols, free_vars, ref_thr)
    baseP = probs(X); K = baseP.shape[1]
    out=[]
    for v in X.columns:
        X1 = X.copy()
        if set(np.unique(X[v])).issubset({0.0,1.0}): X1[v] = 1.0
        else: X1[v] = X1[v] + h
        dP = probs(X1) - baseP
        ame = dP.mean(axis=0)
        for k in range(K):
            out.append({"feature":v, "class":k+1, "AME":float(ame[k])})
    ame_df = pd.DataFrame(out)

    if n_boot>0:
        boots=[]
        for b in range(n_boot):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            Pb = probs(Xb)
            for v in X.columns:
                X1 = Xb.copy()
                if set(np.unique(Xb[v])).issubset({0.0,1.0}): X1[v] = 1.0
                else: X1[v] = X1[v] + h
                dP = probs(X1) - Pb
                ame = dP.mean(axis=0)
                for k in range(K):
                    boots.append({"feature":v,"class":k+1,"AME":float(ame[k]),"boot":b})
        boots = pd.DataFrame(boots)
        cis=[]
        for (f,c), g in boots.groupby(["feature","class"]):
            lo, hi = np.percentile(g["AME"], [2.5, 97.5])
            cis.append({"feature":f,"class":c,"AME_ci_low":lo,"AME_ci_high":hi})
        ame_df = ame_df.merge(pd.DataFrame(cis), on=["feature","class"], how="left")
    return ame_df

# ---------------- 1) 读取 & Ordered Logit & Brant ----------------
X_all, y_all_rank, y_all_orig, df_all, rank_map, inv_rank_map, ordered_original = prep_data(
    FILE_PATH, ordinal_order=ORDINAL_ORDER
)

classes_rank = np.sort(y_all_rank.unique())  # e.g., [1,2,3]（等级标签）

X_tr, X_te, y_tr_rank, y_te_rank = train_test_split(
    X_all, y_all_rank, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all_rank
)

po = OrderedModel(endog=y_tr_rank, exog=X_tr, distr="logit")
po_res = po.fit(method="bfgs", maxiter=1000, disp=False)

brant_global, brant_by = brant_test_like(X_tr, y_tr_rank)

violators = set(brant_by.loc[brant_by["p_value"]<ALPHA_VAR, "feature"].tolist())
for v in FORCE_FREE:
    if v in X_tr.columns: violators.add(v)
free_vars = sorted(list(violators)) or (["age"] if "age" in X_tr.columns else [])
print("将放松非平行的变量：", free_vars)

# ---------------- 2) 叠栈logit：约束 vs 放松 ----------------
long_tr, thr_dum_tr = stack_threshold_data(X_tr, y_tr_rank)
res_con, Zc = refit_with_constraints(long_tr, thr_dum_tr, X_tr)
res_free, Zf, ref_thr, thr_cols = fit_stacked_logit(long_tr, thr_dum_tr, X_tr, free_vars)
res_null, Zn = refit_null(long_tr, thr_dum_tr)

LR, df_lr, p_lr = lr_test(res_free, res_con)
fit_table = pd.DataFrame([
    {"model":"Constrained_PO(stacked)","llf":res_con.llf,"df_model":res_con.df_model,"aic":res_con.aic,"bic":res_con.bic,
     "pseudoR2_vsNull": 1 - (res_con.llf / res_null.llf)},
    {"model":"Partial_PO(Free vars)","llf":res_free.llf,"df_model":res_free.df_model,"aic":res_free.aic,"bic":res_free.bic,
     "pseudoR2_vsNull": 1 - (res_free.llf / res_null.llf)},
])
lr_row = pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}])
coef_by_thr = extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
thr_order = coef_by_thr.query("term=='intercept'")[["threshold","coef"]]
thr_monotone = np.all(np.diff(thr_order["coef"].values) > 0)

# ---------------- 3) 测试集预测 + 指标 ----------------
P_te = predict_gologit(res_free, X_te, thr_cols, free_vars, ref_thr)
y_pred_rank = P_te.argmax(axis=1) + classes_rank.min()

acc, mf1, kappa, mae, brier = ordered_metrics(y_te_rank.to_numpy(), y_pred_rank, P_te)

### NEW: 若你要在报告/表格里显示“原始编码(2/1/3)”，做反映射
y_pred_orig = pd.Series(y_pred_rank).map(inv_rank_map).to_numpy()
y_te_orig   = pd.Series(y_te_rank).map(inv_rank_map).to_numpy()
cm_orig = confusion_matrix(y_te_orig, y_pred_orig, labels=ordered_original)
report_txt = classification_report(y_te_orig, y_pred_orig, digits=3)

# ---------------- 4) 边际效应（可选） ----------------
ame_df = average_marginal_effects_gologit(res_free, X_tr, thr_cols, free_vars, ref_thr, n_boot=N_BOOT_AME)

# ---------------- 5) 导出 Excel ----------------
excel = os.path.join(OUT_DIR, "Gologit_review_ready.xlsx")
with pd.ExcelWriter(excel) as w:
    # A) 统计前提
    pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
    brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
    fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
    lr_row.to_excel(w, "A3_LR_test_partial_vs_constrained", index=False)
    coef_by_thr.to_excel(w, "A4_threshold_level_coefs", index=False)
    pd.DataFrame({"threshold_order":thr_order["threshold"],"intercept":thr_order["coef"],"is_monotone":[thr_monotone]*len(thr_order)}).to_excel(w,"A5_cutpoint_monotonicity",index=False)
    # B) 外推指标（按“原始编码”呈现）
    pd.DataFrame({"metric":["Accuracy","MacroF1","QuadraticKappa","OrdinalMAE","MulticlassBrier"],
                  "value":[acc,mf1,kappa,mae,brier]}).to_excel(w,"B1_test_metrics",index=False)
    pd.DataFrame(cm_orig, index=[f"true_{c}" for c in ordered_original], columns=[f"pred_{c}" for c in ordered_original]).to_excel(w,"B2_confusion_matrix_orig")
    pd.DataFrame({"obs_id":np.arange(len(y_te_orig)),"y_test_orig":y_te_orig,"y_pred_orig":y_pred_orig}).to_excel(w,"B3_test_predictions_orig",index=False)
    ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)

# ---------------- 6) 控制台摘要 ----------------
print("=== 统计前提（结构） ===")
print(f"Brant 全局: {brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}")
print("将放松的变量：", list(free_vars))
print("\n约束 vs 放松 的拟合：\n", fit_table.to_string(index=False))
print("\n似然比检验（放松 vs 约束）：\n", lr_row.to_string(index=False))
print("\n阈值截距是否单调递增？ ->", thr_monotone)

print("\n=== 外推（测试集，原始编码显示） ===")
print(f"Accuracy={acc:.3f}, Macro-F1={mf1:.3f}, Quadratic κ={kappa:.3f}, Ordinal MAE(等级)={mae:.3f}, Brier={brier:.4f}")
print("混淆矩阵(原始编码):\n", cm_orig)
print("\n分类报告(原始编码):\n", report_txt)
print("\nExcel 文件：", excel)


将放松非平行的变量： ['age', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_3.0', 'trajet_5.0']
=== 统计前提（结构） ===
Brant 全局: 232.11, df=32, p=0
将放松的变量： ['age', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_3.0', 'trajet_5.0']

约束 vs 放松 的拟合：
                   model          llf  df_model          aic          bic  pseudoR2_vsNull
Constrained_PO(stacked) -8042.762298      33.0 16153.524596 16422.983064         0.376277
  Partial_PO(Free vars) -7931.105740      48.0 15960.211481 16348.548684         0.384936

似然比检验（放松 vs 约束）：
    LR_stat  df  p_value
223.313116  15      0.0

阈值截距是否单调递增？ -> False

=== 外推（测试集，原始编码显示） ===
Accuracy=0.639, Macro-F1=0.648, Quadratic κ=0.648, Ordinal MAE(等级)=0.399, Brier=0.1475
混淆矩阵(原始编码):
 [[712 271  23]
 [450 348  41]
 [ 74  64 573]]

分类报告(原始编码):
               precision    recall  f1-score   support

       

/tmp/ipython-input-3680401782.py:302: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
/tmp/ipython-input-3680401782.py:303: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
/tmp/ipython-input-3680401782.py:304: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
/tmp/ipython-input-3680401782.py:305: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the a

In [ ]:
# ============================================================
# Generalized Ordered Logit (Partial Proportional Odds) – Reviewer-ready
# + age × secu 交互：阈值系数、LR检验（含交互 vs 不含交互）、AME(age|secu)
# 支持自定义有序顺序（如 2<1<3），训练用“等级化标签”，对外显示可反映射回原编码

#####检查age和secu交互项的代码
# ============================================================

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
from scipy.special import expit
from scipy.stats import chi2

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH   = "//content/crash_data(final).csv"   # ← 改成你的CSV路径
TARGET      = "grav_balanced"
OUT_DIR     = "/content/gologit_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE   = 0.2
RANDOM_STATE= 42
RARE_MIN    = 150
ALPHA_VAR   = 0.05
FORCE_FREE  = ["age"]
N_BOOT_AME  = 0

### 业务顺序（从低到高的严重度）。你的场景是 2(无伤)<1(轻伤)<3(重伤)：
ORDINAL_ORDER = [2, 1, 3]   # 如已是自然递增(1<2<3)可设为 None 或 [1,2,3]

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str)
    vc = s.value_counts(dropna=True)
    return s.where(~s.isin(vc[vc<k].index), "__OTHER__")

def _build_rank_maps(y_series, ordinal_order):
    """
    根据业务顺序生成 rank_map 与 inv_rank_map:
      原标签 -> 等级(1..K)； 等级 -> 原标签
    """
    if ordinal_order is None:
        ordered = sorted(pd.unique(y_series))
    else:
        ordered = list(ordinal_order)
    rank_map = {v: i+1 for i, v in enumerate(ordered)}
    inv_rank_map = {r: v for v, r in rank_map.items()}
    return rank_map, inv_rank_map, ordered

def prep_data(path, ordinal_order=None):
    """
    返回：
      X（含 age × secu_* 交互）、y_rank、y_orig、df、rank_map、inv_rank_map、ordered_original、secu_dummies
    """
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0  = pd.read_csv(path)
    use  = [c for c in need if c in df0.columns]
    if "grav_balanced" not in use:
        raise ValueError("缺少 grav_balanced")
    df = df0[use].dropna().copy()

    # 目标为整数
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

    # 映射并把 y 转为 1..K 的等级
    rank_map, inv_rank_map, ordered_original = _build_rank_maps(df[TARGET], ordinal_order)
    y_orig = df[TARGET].copy()
    y_rank = df[TARGET].map(rank_map).astype(int)

    # 合并稀有水平
    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        df[c] = merge_rare(df[c], RARE_MIN)

    # 数值
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = (df["age"] - df["age"].mean())/(df["age"].std(ddof=0) if df["age"].std(ddof=0)!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)

    # one-hot
    X_cat = pd.get_dummies(df[["secu","obs","plan","lum","sexe","trajet"]], drop_first=True, dtype=float)
    X_num = df[["age"] + (["age_secu"] if "age_secu" in df.columns else [])].astype(float)

    # === age × secu_* 交互项 ===
    secu_dummies = [c for c in X_cat.columns if c.startswith("secu_")]
    X_int = pd.DataFrame(index=df.index)
    for c in secu_dummies:
        X_int[f"age_x_{c}"] = X_num["age"] * X_cat[c]

    X = pd.concat([X_num, X_cat, X_int], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]   # 去零方差列

    return X, y_rank, y_orig, df, rank_map, inv_rank_map, ordered_original, secu_dummies

def brant_test_like(X, y_rank):
    """
    简化版 Brant：对各阈值拟二分类 Logit，比较斜率是否随阈值变化
    """
    levels = np.sort(y_rank.unique()); th = levels[:-1]
    betas, covs = [], []
    for k in th:
        yk = (y_rank > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=400)
        b = res.params.drop("const"); V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values); covs.append(V.values)
    betas = np.stack(betas, axis=0)  # (K-1, p)
    p = betas.shape[1]; K1 = betas.shape[0]

    diffs, Vblk = [], []
    for k in range(1, K1):
        diffs.append(betas[k]-betas[0])
        Vblk.append(covs[k]+covs[0])
    d = np.concatenate(diffs)
    V = np.zeros(((K1-1)*p, (K1-1)*p))
    for i in range(K1-1):
        V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    chi2_stat = float(d.T @ Vinv @ d); df_w = d.size; p_global = 1.0 - chi2.cdf(chi2_stat, df_w)
    global_row = pd.DataFrame([{"scope":"global","chi2_stat":chi2_stat,"df":df_w,"p_value":p_global}])

    rows=[]
    for j,name in enumerate(X.columns):
        dj = np.array([betas[k,j]-betas[0,j] for k in range(1,K1)])
        Vj = np.diag([covs[k][j,j]+covs[0][j,j] for k in range(1,K1)])
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj); dfj = dj.size; pj = 1.0 - chi2.cdf(statj, dfj)
        rows.append({"feature":name,"chi2_stat":statj,"df":dfj,"p_value":pj})
    return global_row, pd.DataFrame(rows)

def stack_threshold_data(X, y_rank):
    classes = np.sort(y_rank.unique()); th = classes[:-1]
    n = len(y_rank); rep = len(th)
    long = pd.DataFrame({
        "obs_id": np.repeat(np.arange(n), rep),
        "thr":    np.tile(th, n),
        "y_bin":  (np.repeat(y_rank.to_numpy(), rep) > np.tile(th, n)).astype(int),
    })
    Xrep = np.repeat(X.to_numpy(), rep, axis=0)
    long = pd.concat([long, pd.DataFrame(Xrep, columns=X.columns)], axis=1)
    thr_dum = pd.get_dummies(long["thr"].astype(str), drop_first=False, dtype=int)
    thr_dum.columns = [f"thr_{c}" for c in thr_dum.columns]
    return long.reset_index(drop=True), thr_dum

def fit_stacked_logit(long_df, thr_dum, X, free_vars):
    Z = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    thr_cols = [c for c in thr_dum.columns]
    ref_thr = thr_cols[0]
    for v in free_vars:
        for t in thr_cols[1:]:
            Z[f"{v}__x__{t}"] = long_df[v].to_numpy() * thr_dum[t].to_numpy()
    res = sm.Logit(long_df["y_bin"], Z).fit(disp=False, maxiter=400, cov_type="cluster",
                                            cov_kwds={"groups": long_df["obs_id"]})
    return res, Z, ref_thr, thr_cols

def refit_with_constraints(long_df, thr_dum, X):
    Zc = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    resC = sm.Logit(long_df["y_bin"], Zc).fit(disp=False, maxiter=400, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resC, Zc

def refit_null(long_df, thr_dum):
    Zn = thr_dum.copy()
    resN = sm.Logit(long_df["y_bin"], Zn).fit(disp=False, maxiter=200, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resN, Zn

def lr_test(res_full, res_small):
    LR = 2*(res_full.llf - res_small.llf)
    df = res_full.df_model - res_small.df_model
    p = 1.0 - chi2.cdf(LR, df if df>0 else 1)
    return LR, int(df), float(p)

def extract_threshold_coefs(res, Xcols, thr_cols, free_vars, ref_thr):
    coefs = res.params; cov = res.cov_params()
    rows=[]
    for t in thr_cols:
        b0 = coefs[t]; se0 = np.sqrt(cov.loc[t,t]); rows.append(
            {"threshold":t, "term":"intercept", "coef":b0, "se":se0,
             "ci_low":b0-1.96*se0, "ci_high":b0+1.96*se0, "OR":np.nan}
        )
        for v in Xcols:
            if v in free_vars:
                if t==ref_thr:
                    b = coefs.get(v, 0.0); var = cov.loc[v,v]
                else:
                    dv = f"{v}__x__{t}"
                    b = coefs.get(v, 0.0) + coefs.get(dv, 0.0)
                    var = cov.loc[v,v] + cov.loc[dv,dv] + 2*cov.loc[v,dv]
            else:
                b = coefs.get(v, 0.0); var = cov.loc[v,v]
            se = np.sqrt(var)
            rows.append({"threshold":t, "term":v, "coef":b, "se":se,
                         "ci_low":b-1.96*se, "ci_high":b+1.96*se, "OR":np.exp(b)})
    return pd.DataFrame(rows)

def predict_gologit(res, X, thr_cols, free_vars, ref_thr):
    """
    针对三分类（K=3）的概率预测：
      p1 = 1 - P(y>1)
      p3 = P(y>2)
      p2 = P(y>1) - P(y>2)
    """
    coefs = res.params
    Pgt = []
    for t in thr_cols:
        eta = coefs[t]
        # 固定斜率的变量
        for v in X.columns:
            if v not in free_vars:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
        # 放松斜率的变量
        for v in free_vars:
            if t == ref_thr:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
            else:
                eta += X[v].to_numpy() * (coefs.get(v, 0.0) + coefs.get(f"{v}__x__{t}", 0.0))
        Pgt.append(expit(eta))
    Pgt = np.vstack(Pgt)                # (K-1, n)
    p1 = 1.0 - Pgt[0]; p3 = Pgt[-1]; p2 = Pgt[0] - Pgt[-1]
    P = np.vstack([p1, p2, p3]).T
    P = np.clip(P, 1e-12, 1-1e-12); P = P / P.sum(axis=1, keepdims=True)
    return P

def ordered_metrics(y_true_rank, y_pred_rank, prob):
    acc = accuracy_score(y_true_rank, y_pred_rank)
    mf1 = f1_score(y_true_rank, y_pred_rank, average="macro")
    kappa = cohen_kappa_score(y_true_rank, y_pred_rank, weights="quadratic")
    mae = np.mean(np.abs(y_true_rank - y_pred_rank))
    K = prob.shape[1]
    Y = np.eye(K)[(y_true_rank-1)]
    brier = np.mean(np.sum((prob - Y)**2, axis=1)) / K
    return acc, mf1, kappa, mae, brier

def average_marginal_effects_gologit(res, X, thr_cols, free_vars, ref_thr, h=1e-4, n_boot=0, seed=123):
    rng = np.random.default_rng(seed)
    def probs(ex): return predict_gologit(res, ex, thr_cols, free_vars, ref_thr)
    baseP = probs(X); K = baseP.shape[1]
    out=[]
    for v in X.columns:
        X1 = X.copy()
        if set(np.unique(X[v])).issubset({0.0,1.0}):
            X1[v] = 1.0
        else:
            X1[v] = X1[v] + h
        dP = probs(X1) - baseP
        ame = dP.mean(axis=0)
        for k in range(K):
            out.append({"feature":v, "class":k+1, "AME":float(ame[k])})
    ame_df = pd.DataFrame(out)

    if n_boot>0:
        boots=[]
        for b in range(n_boot):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            Pb = probs(Xb)
            for v in X.columns:
                X1 = Xb.copy()
                if set(np.unique(Xb[v])).issubset({0.0,1.0}):
                    X1[v] = 1.0
                else:
                    X1[v] = X1[v] + h
                dP = probs(X1) - Pb
                ame = dP.mean(axis=0)
                for k in range(K):
                    boots.append({"feature":v,"class":k+1,"AME":float(ame[k]),"boot":b})
        boots = pd.DataFrame(boots)
        cis=[]
        for (f,c), g in boots.groupby(["feature","class"]):
            lo, hi = np.percentile(g["AME"], [2.5, 97.5])
            cis.append({"feature":f,"class":c,"AME_ci_low":lo,"AME_ci_high":hi})
        ame_df = ame_df.merge(pd.DataFrame(cis), on=["feature","class"], how="left")
    return ame_df

def ame_age_within_secu(res, X, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4):
    """
    在不同 secu 水平下，age 增加 h 时，三类概率的平均变化（AME）。
    逻辑：复制 X -> X1; 对所有样本 X1['age'] += h；
         若样本属于某个 secu_*==1，则对应的 age_x_secu_* 同步 += h（基类无交互增量）。
    返回 DataFrame: secu_level, class(1..K), AME_age
    """
    baseP = predict_gologit(res, X, thr_cols, free_vars, ref_thr)
    out=[]
    # 基类（被 drop 的 secu 参考水平）
    Xb = X.copy()
    Xb["age"] = Xb["age"] + h
    Pb = predict_gologit(res, Xb, thr_cols, free_vars, ref_thr)
    ame_base = (Pb - baseP).mean(axis=0)
    for k in range(baseP.shape[1]):
        out.append({"secu_level":"__BASE__", "class":k+1, "AME_age":float(ame_base[k])})

    # 其它 one-hot secu 水平
    for s in secu_dummies:
        X1 = X.copy()
        X1["age"] = X1["age"] + h
        inter = f"age_x_{s}"
        if inter in X1.columns:
            idx = X1[s]==1.0
            X1.loc[idx, inter] = X1.loc[idx, inter] + h
        P1 = predict_gologit(res, X1, thr_cols, free_vars, ref_thr)
        ame = (P1 - baseP).mean(axis=0)
        for k in range(baseP.shape[1]):
            out.append({"secu_level":s.replace("secu_",""), "class":k+1, "AME_age":float(ame[k])})
    return pd.DataFrame(out)

# ---------------- 1) 读取 & Ordered Logit & Brant ----------------
X_all, y_all_rank, y_all_orig, df_all, rank_map, inv_rank_map, ordered_original, secu_dummies = prep_data(
    FILE_PATH, ordinal_order=ORDINAL_ORDER
)

classes_rank = np.sort(y_all_rank.unique())  # e.g., [1,2,3]（等级标签）

X_tr, X_te, y_tr_rank, y_te_rank = train_test_split(
    X_all, y_all_rank, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all_rank
)

# 可选：标准 OrderedModel（用于对照，不用于预测）
po = OrderedModel(endog=y_tr_rank, exog=X_tr, distr="logit")
po_res = po.fit(method="bfgs", maxiter=1000, disp=False)

# Brant-like 检验
brant_global, brant_by = brant_test_like(X_tr, y_tr_rank)

# ---------------- 放松变量选择（含 age × secu 交互） ----------------
violators = set(brant_by.loc[brant_by["p_value"]<ALPHA_VAR, "feature"].tolist())
for v in FORCE_FREE:
    if v in X_tr.columns: violators.add(v)

# 交互项名列表
age_secu_inters = [c for c in X_tr.columns if c.startswith("age_x_secu_")]
# 建议：交互项优先放入 free_vars
for v in age_secu_inters:
    violators.add(v)

free_vars = sorted(list(violators)) or (["age"] if "age" in X_tr.columns else [])
print("将放松非平行的变量（含 age×secu 交互）：", free_vars)

# ---------------- 2) 叠栈logit：约束 vs 放松 ----------------
long_tr, thr_dum_tr = stack_threshold_data(X_tr, y_tr_rank)
res_con, Zc = refit_with_constraints(long_tr, thr_dum_tr, X_tr)
res_free, Zf, ref_thr, thr_cols = fit_stacked_logit(long_tr, thr_dum_tr, X_tr, free_vars)
res_null, Zn = refit_null(long_tr, thr_dum_tr)

LR, df_lr, p_lr = lr_test(res_free, res_con)
fit_table = pd.DataFrame([
    {"model":"Constrained_PO(stacked)","llf":res_con.llf,"df_model":res_con.df_model,"aic":res_con.aic,"bic":res_con.bic,
     "pseudoR2_vsNull": 1 - (res_con.llf / res_null.llf)},
    {"model":"Partial_PO(Free vars)","llf":res_free.llf,"df_model":res_free.df_model,"aic":res_free.aic,"bic":res_free.bic,
     "pseudoR2_vsNull": 1 - (res_free.llf / res_null.llf)},
])
lr_row = pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}])
coef_by_thr = extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
thr_order = coef_by_thr.query("term=='intercept'")[["threshold","coef"]]
thr_monotone = np.all(np.diff(thr_order["coef"].values) > 0)

# ---------------- 2.5) “含交互 vs 不含交互”的 LR 检验 ----------------
X_tr_noint = X_tr.drop(columns=[c for c in X_tr.columns if c.startswith("age_x_secu_")], errors="ignore")
long_tr_noint, thr_dum_tr_noint = stack_threshold_data(X_tr_noint, y_tr_rank)
free_vars_noint = [v for v in free_vars if v in X_tr_noint.columns]
res_con_noint, Zc_noint = refit_with_constraints(long_tr_noint, thr_dum_tr_noint, X_tr_noint)
res_free_noint, Zf_noint, ref_thr_noint, thr_cols_noint = fit_stacked_logit(long_tr_noint, thr_dum_tr_noint, X_tr_noint, free_vars_noint)
res_null_noint, Zn_noint = refit_null(long_tr_noint, thr_dum_tr_noint)

LR_int, df_int, p_int = lr_test(res_free, res_free_noint)  # 含交互 vs 不含交互（均为“放松”模型）
lr_interaction_row = pd.DataFrame([{"LR_stat":LR_int,"df":df_int,"p_value":p_int}])

# ---------------- 3) 测试集预测 + 指标 ----------------
P_te = predict_gologit(res_free, X_te, thr_cols, free_vars, ref_thr)
y_pred_rank = P_te.argmax(axis=1) + classes_rank.min()

acc, mf1, kappa, mae, brier = ordered_metrics(y_te_rank.to_numpy(), y_pred_rank, P_te)

# 原始编码映射输出
y_pred_orig = pd.Series(y_pred_rank).map(inv_rank_map).to_numpy()
y_te_orig   = pd.Series(y_te_rank).map(inv_rank_map).to_numpy()
cm_orig = confusion_matrix(y_te_orig, y_pred_orig, labels=ordered_original)
report_txt = classification_report(y_te_orig, y_pred_orig, digits=3)

# ---------------- 4) 边际效应（总表 + age 在各 secu 水平下） ----------------
ame_df = average_marginal_effects_gologit(res_free, X_tr, thr_cols, free_vars, ref_thr, n_boot=N_BOOT_AME)
ame_age_secu = ame_age_within_secu(res_free, X_tr, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4)

# ---------------- 5) 导出 Excel ----------------
excel = os.path.join(OUT_DIR, "Gologit_review_ready.xlsx")
with pd.ExcelWriter(excel) as w:
    # A) 统计前提
    pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
    brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
    fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
    lr_row.to_excel(w, "A3_LR_test_partial_vs_constrained", index=False)
    coef_by_thr.to_excel(w, "A4_threshold_level_coefs", index=False)
    pd.DataFrame({"threshold_order":thr_order["threshold"],"intercept":thr_order["coef"],"is_monotone":[thr_monotone]*len(thr_order)}).to_excel(w,"A5_cutpoint_monotonicity",index=False)
    # B) 外推指标（按“原始编码”呈现）
    pd.DataFrame({"metric":["Accuracy","MacroF1","QuadraticKappa","OrdinalMAE","MulticlassBrier"],
                  "value":[acc,mf1,kappa,mae,brier]}).to_excel(w,"B1_test_metrics",index=False)
    pd.DataFrame(cm_orig, index=[f"true_{c}" for c in ordered_original], columns=[f"pred_{c}" for c in ordered_original]).to_excel(w,"B2_confusion_matrix_orig")
    pd.DataFrame({"obs_id":np.arange(len(y_te_orig)),"y_test_orig":y_te_orig,"y_pred_orig":y_pred_orig}).to_excel(w,"B3_test_predictions_orig",index=False)
    ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)
    # C) 交互的显著性与效应
    lr_interaction_row.to_excel(w, "C1_LR_interaction_vs_base", index=False)
    (coef_by_thr[coef_by_thr["term"].str.startswith("age_x_secu_")]
        .to_excel(w, "C2_interaction_coefs_by_threshold", index=False))
    ame_age_secu.to_excel(w, "C3_AME_age_by_secu", index=False)

# ---------------- 6) 控制台摘要 ----------------
print("=== 统计前提（结构） ===")
print(f"Brant 全局: {brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}")
print("将放松的变量（含 age×secu 交互）：", list(free_vars))
print("\n约束 vs 放松 的拟合：\n", fit_table.to_string(index=False))
print("\n似然比检验（放松 vs 约束）：\n", lr_row.to_string(index=False))
print("\n含交互 vs 不含交互（放松模型）：\n", lr_interaction_row.to_string(index=False))
print("\n阈值截距是否单调递增？ ->", thr_monotone)

print("\n=== 外推（测试集，原始编码显示） ===")
print(f"Accuracy={acc:.3f}, Macro-F1={mf1:.3f}, Quadratic κ={kappa:.3f}, Ordinal MAE(等级)={mae:.3f}, Brier={brier:.4f}")
print("混淆矩阵(原始编码):\n", cm_orig)
print("\n分类报告(原始编码):\n", report_txt)
print("\nExcel 文件：", excel)


将放松非平行的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']


/tmp/ipython-input-1807882265.py:387: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
/tmp/ipython-input-1807882265.py:388: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
/tmp/ipython-input-1807882265.py:389: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
/tmp/ipython-input-1807882265.py:390: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the a

=== 统计前提（结构） ===
Brant 全局: 234.51, df=35, p=0
将放松的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']

约束 vs 放松 的拟合：
                   model          llf  df_model          aic          bic  pseudoR2_vsNull
Constrained_PO(stacked) -8040.191821      36.0 16154.383641 16447.617856         0.376476
  Partial_PO(Free vars) -7928.838286      53.0 15965.676573 16393.640021         0.385112

似然比检验（放松 vs 约束）：
    LR_stat  df  p_value
222.707068  17      0.0

含交互 vs 不含交互（放松模型）：
  LR_stat  df  p_value
 7.60681   6 0.268347

阈值截距是否单调递增？ -> False

=== 外推（测试集，原始编码显示） ===
Accuracy=0.639, Macro-F1=0.647, Quadratic κ=0.646, Ordinal MAE(等级)=0.400, Brier=0.1475
混淆矩阵(原始编码):
 [[719 263  24]
 [457 342  40]
 [ 77  61 573]]

分类报告(原始编码):
               precision    recall  f1-score   support

           1      0.514     0.408     0.454       839
       

/tmp/ipython-input-1807882265.py:398: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)
/tmp/ipython-input-1807882265.py:400: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  lr_interaction_row.to_excel(w, "C1_LR_interaction_vs_base", index=False)
/tmp/ipython-input-1807882265.py:402: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  .to_excel(w, "C2_interaction_coefs_by_threshold", index=False))
/tmp/ipython-input-1807882265.py:403: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  ame_age_secu.to_excel(w, "C3_AME_age_by_secu", index=False)


In [ ]:
# ============================================================
# Generalized Ordered Logit (Partial Proportional Odds) – Reviewer-ready
# + age × secu 交互；含交互 vs 不含交互 LR；AME(age|secu)
# + NEW: 导出“全量特征的 AME+95%CI”为 CSV（长表 & 论文式宽表）
# ============================================================

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
from scipy.special import expit
from scipy.stats import chi2
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH   = "//content/crash_data(final).csv"   # ← 改成你的CSV路径
TARGET      = "grav_balanced"
OUT_DIR     = "/content/gologit_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE   = 0.2
RANDOM_STATE= 42
RARE_MIN    = 150
ALPHA_VAR   = 0.05
FORCE_FREE  = ["age"]
N_BOOT_AME  = 500     # ← 设为>0 会给 AME 自动计算 95% CI
ORDINAL_ORDER = [2, 1, 3]   # 你的业务顺序：2(无伤)<1(轻伤)<3(重伤)

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str); vc = s.value_counts(dropna=True)
    return s.where(~s.isin(vc[vc<k].index), "__OTHER__")

def _build_rank_maps(y_series, ordinal_order):
    if ordinal_order is None: ordered = sorted(pd.unique(y_series))
    else: ordered = list(ordinal_order)
    rank_map = {v: i+1 for i, v in enumerate(ordered)}
    inv_rank_map = {r: v for v, r in rank_map.items()}
    return rank_map, inv_rank_map, ordered

def prep_data(path, ordinal_order=None):
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0  = pd.read_csv(path)
    use  = [c for c in need if c in df0.columns]
    if "grav_balanced" not in use: raise ValueError("缺少 grav_balanced")
    df = df0[use].dropna().copy()

    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)
    rank_map, inv_rank_map, ordered_original = _build_rank_maps(df[TARGET], ordinal_order)
    y_orig = df[TARGET].copy()
    y_rank = df[TARGET].map(rank_map).astype(int)

    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        df[c] = merge_rare(df[c], RARE_MIN)

    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = (df["age"] - df["age"].mean())/(df["age"].std(ddof=0) if df["age"].std(ddof=0)!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)

    X_cat = pd.get_dummies(df[["secu","obs","plan","lum","sexe","trajet"]], drop_first=True, dtype=float)
    X_num = df[["age"] + (["age_secu"] if "age_secu" in df.columns else [])].astype(float)

    # age × secu_* 交互
    secu_dummies = [c for c in X_cat.columns if c.startswith("secu_")]
    X_int = pd.DataFrame(index=df.index)
    for c in secu_dummies:
        X_int[f"age_x_{c}"] = X_num["age"] * X_cat[c]

    X = pd.concat([X_num, X_cat, X_int], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]
    return X, y_rank, y_orig, df, rank_map, inv_rank_map, ordered_original, secu_dummies

def brant_test_like(X, y_rank):
    levels = np.sort(y_rank.unique()); th = levels[:-1]
    betas, covs = [], []
    for k in th:
        yk = (y_rank > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=400)
        b = res.params.drop("const"); V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values); covs.append(V.values)
    betas = np.stack(betas, axis=0)
    p = betas.shape[1]; K1 = betas.shape[0]

    diffs, Vblk = [], []
    for k in range(1,K1):
        diffs.append(betas[k]-betas[0]); Vblk.append(covs[k]+covs[0])
    d = np.concatenate(diffs)
    V = np.zeros(((K1-1)*p,(K1-1)*p))
    for i in range(K1-1): V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    from scipy.stats import chi2 as _chi2
    chi2_stat = float(d.T @ Vinv @ d); df_w = d.size; p_global = 1.0 - _chi2.cdf(chi2_stat, df_w)
    global_row = pd.DataFrame([{"scope":"global","chi2_stat":chi2_stat,"df":df_w,"p_value":p_global}])

    rows=[]
    for j,name in enumerate(X.columns):
        dj = np.array([betas[k,j]-betas[0,j] for k in range(1,K1)])
        Vj = np.diag([covs[k][j,j]+covs[0][j,j] for k in range(1,K1)])
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj); dfj = dj.size; pj = 1.0 - _chi2.cdf(statj, dfj)
        rows.append({"feature":name,"chi2_stat":statj,"df":dfj,"p_value":pj})
    return global_row, pd.DataFrame(rows)

def stack_threshold_data(X, y_rank):
    classes = np.sort(y_rank.unique()); th = classes[:-1]
    n = len(y_rank); rep = len(th)
    long = pd.DataFrame({
        "obs_id": np.repeat(np.arange(n), rep),
        "thr":    np.tile(th, n),
        "y_bin":  (np.repeat(y_rank.to_numpy(), rep) > np.tile(th, n)).astype(int),
    })
    Xrep = np.repeat(X.to_numpy(), rep, axis=0)
    long = pd.concat([long, pd.DataFrame(Xrep, columns=X.columns)], axis=1)
    thr_dum = pd.get_dummies(long["thr"].astype(str), drop_first=False, dtype=int)
    thr_dum.columns = [f"thr_{c}" for c in thr_dum.columns]
    return long.reset_index(drop=True), thr_dum

def fit_stacked_logit(long_df, thr_dum, X, free_vars):
    Z = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    thr_cols = [c for c in thr_dum.columns]; ref_thr = thr_cols[0]
    for v in free_vars:
        for t in thr_cols[1:]:
            Z[f"{v}__x__{t}"] = long_df[v].to_numpy() * thr_dum[t].to_numpy()
    res = sm.Logit(long_df["y_bin"], Z).fit(disp=False, maxiter=400, cov_type="cluster",
                                            cov_kwds={"groups": long_df["obs_id"]})
    return res, Z, ref_thr, thr_cols

def refit_with_constraints(long_df, thr_dum, X):
    Zc = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    resC = sm.Logit(long_df["y_bin"], Zc).fit(disp=False, maxiter=400, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resC, Zc

def refit_null(long_df, thr_dum):
    Zn = thr_dum.copy()
    resN = sm.Logit(long_df["y_bin"], Zn).fit(disp=False, maxiter=200, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resN, Zn

def lr_test(res_full, res_small):
    LR = 2*(res_full.llf - res_small.llf)
    df = res_full.df_model - res_small.df_model
    p = 1.0 - chi2.cdf(LR, df if df>0 else 1)
    return LR, int(df), float(p)

def extract_threshold_coefs(res, Xcols, thr_cols, free_vars, ref_thr):
    coefs = res.params; cov = res.cov_params()
    rows=[]
    for t in thr_cols:
        b0 = coefs[t]; se0 = np.sqrt(cov.loc[t,t])
        rows.append({"threshold":t,"term":"intercept","coef":b0,"se":se0,
                     "ci_low":b0-1.96*se0,"ci_high":b0+1.96*se0,"OR":np.nan})
        for v in Xcols:
            if v in free_vars:
                if t==ref_thr:
                    b = coefs.get(v, 0.0); var = cov.loc[v,v]
                else:
                    dv = f"{v}__x__{t}"
                    b = coefs.get(v, 0.0)+coefs.get(dv, 0.0)
                    var = cov.loc[v,v]+cov.loc[dv,dv]+2*cov.loc[v,dv]
            else:
                b = coefs.get(v, 0.0); var = cov.loc[v,v]
            se = np.sqrt(var)
            rows.append({"threshold":t,"term":v,"coef":b,"se":se,
                         "ci_low":b-1.96*se,"ci_high":b+1.96*se,"OR":np.exp(b)})
    return pd.DataFrame(rows)

def predict_gologit(res, X, thr_cols, free_vars, ref_thr):
    coefs = res.params; Pgt=[]
    for t in thr_cols:
        eta = coefs[t]
        for v in X.columns:
            if v not in free_vars:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
        for v in free_vars:
            if t==ref_thr:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
            else:
                eta += X[v].to_numpy() * (coefs.get(v, 0.0)+coefs.get(f"{v}__x__{t}", 0.0))
        Pgt.append(expit(eta))
    Pgt = np.vstack(Pgt)
    p1 = 1.0 - Pgt[0]; p3 = Pgt[-1]; p2 = Pgt[0]-Pgt[-1]
    P = np.vstack([p1,p2,p3]).T
    P = np.clip(P, 1e-12, 1-1e-12); P = P / P.sum(axis=1, keepdims=True)
    return P

def ordered_metrics(y_true_rank, y_pred_rank, prob):
    acc = accuracy_score(y_true_rank, y_pred_rank)
    mf1 = f1_score(y_true_rank, y_pred_rank, average="macro")
    kappa = cohen_kappa_score(y_true_rank, y_pred_rank, weights="quadratic")
    mae = np.mean(np.abs(y_true_rank - y_pred_rank))
    K = prob.shape[1]; Y = np.eye(K)[(y_true_rank-1)]
    brier = np.mean(np.sum((prob - Y)**2, axis=1)) / K
    return acc, mf1, kappa, mae, brier

def average_marginal_effects_gologit(res, X, thr_cols, free_vars, ref_thr, h=1e-4, n_boot=0, seed=123):
    rng = np.random.default_rng(seed)
    def probs(ex): return predict_gologit(res, ex, thr_cols, free_vars, ref_thr)
    baseP = probs(X); K = baseP.shape[1]
    out=[]
    for v in X.columns:
        X1 = X.copy()
        if set(np.unique(X[v])).issubset({0.0,1.0}): X1[v] = 1.0
        else: X1[v] = X1[v] + h
        dP = probs(X1) - baseP
        ame = dP.mean(axis=0)
        for k in range(K):
            out.append({"feature":v,"class":k+1,"AME":float(ame[k])})
    ame_df = pd.DataFrame(out)

    if n_boot>0:
        boots=[]
        for b in range(n_boot):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            Pb = probs(Xb)
            for v in X.columns:
                X1 = Xb.copy()
                if set(np.unique(Xb[v])).issubset({0.0,1.0}): X1[v] = 1.0
                else: X1[v] = X1[v] + h
                dP = probs(X1) - Pb
                ame = dP.mean(axis=0)
                for k in range(K):
                    boots.append({"feature":v,"class":k+1,"AME":float(ame[k]),"boot":b})
        boots = pd.DataFrame(boots)
        cis=[]
        for (f,c), g in boots.groupby(["feature","class"]):
            lo, hi = np.percentile(g["AME"], [2.5, 97.5])
            cis.append({"feature":f,"class":c,"AME_ci_low":lo,"AME_ci_high":hi})
        ame_df = ame_df.merge(pd.DataFrame(cis), on=["feature","class"], how="left")
    return ame_df

def ame_age_within_secu(res, X, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4):
    baseP = predict_gologit(res, X, thr_cols, free_vars, ref_thr)
    out=[]
    Xb = X.copy(); Xb["age"] = Xb["age"] + h
    Pb = predict_gologit(res, Xb, thr_cols, free_vars, ref_thr)
    ame_base = (Pb - baseP).mean(axis=0)
    for k in range(baseP.shape[1]):
        out.append({"secu_level":"__BASE__", "class":k+1, "AME_age":float(ame_base[k])})
    for s in secu_dummies:
        X1 = X.copy(); X1["age"] = X1["age"] + h
        inter = f"age_x_{s}"
        if inter in X1.columns:
            idx = X1[s]==1.0
            X1.loc[idx, inter] = X1.loc[idx, inter] + h
        P1 = predict_gologit(res, X1, thr_cols, free_vars, ref_thr)
        ame = (P1 - baseP).mean(axis=0)
        for k in range(baseP.shape[1]):
            out.append({"secu_level":s.replace("secu_",""), "class":k+1, "AME_age":float(ame[k])})
    return pd.DataFrame(out)

# ---------------- 1) 读取 & Ordered Logit & Brant ----------------
X_all, y_all_rank, y_all_orig, df_all, rank_map, inv_rank_map, ordered_original, secu_dummies = prep_data(
    FILE_PATH, ordinal_order=ORDINAL_ORDER
)
classes_rank = np.sort(y_all_rank.unique())

X_tr, X_te, y_tr_rank, y_te_rank = train_test_split(
    X_all, y_all_rank, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all_rank
)

po = OrderedModel(endog=y_tr_rank, exog=X_tr, distr="logit")
po_res = po.fit(method="bfgs", maxiter=1000, disp=False)

brant_global, brant_by = brant_test_like(X_tr, y_tr_rank)

# ---------------- 放松变量选择（含 age × secu 交互） ----------------
violators = set(brant_by.loc[brant_by["p_value"]<ALPHA_VAR, "feature"].tolist())
for v in FORCE_FREE:
    if v in X_tr.columns: violators.add(v)
age_secu_inters = [c for c in X_tr.columns if c.startswith("age_x_secu_")]
for v in age_secu_inters: violators.add(v)
free_vars = sorted(list(violators)) or (["age"] if "age" in X_tr.columns else [])
print("将放松非平行的变量（含 age×secu 交互）：", free_vars)

# ---------------- 2) 叠栈logit：约束 vs 放松 ----------------
long_tr, thr_dum_tr = stack_threshold_data(X_tr, y_tr_rank)
res_con, Zc = refit_with_constraints(long_tr, thr_dum_tr, X_tr)
res_free, Zf, ref_thr, thr_cols = fit_stacked_logit(long_tr, thr_dum_tr, X_tr, free_vars)
res_null, Zn = refit_null(long_tr, thr_dum_tr)

LR, df_lr, p_lr = lr_test(res_free, res_con)
fit_table = pd.DataFrame([
    {"model":"Constrained_PO(stacked)","llf":res_con.llf,"df_model":res_con.df_model,"aic":res_con.aic,"bic":res_con.bic,
     "pseudoR2_vsNull": 1 - (res_con.llf / res_null.llf)},
    {"model":"Partial_PO(Free vars)","llf":res_free.llf,"df_model":res_free.df_model,"aic":res_free.aic,"bic":res_free.bic,
     "pseudoR2_vsNull": 1 - (res_free.llf / res_null.llf)},
])
lr_row = pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}])
coef_by_thr = extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
thr_order = coef_by_thr.query("term=='intercept'")[["threshold","coef"]]
thr_monotone = np.all(np.diff(thr_order["coef"].values) > 0)

# ---------------- 2.5) “含交互 vs 不含交互”的 LR 检验 ----------------
X_tr_noint = X_tr.drop(columns=[c for c in X_tr.columns if c.startswith("age_x_secu_")], errors="ignore")
long_tr_noint, thr_dum_tr_noint = stack_threshold_data(X_tr_noint, y_tr_rank)
free_vars_noint = [v for v in free_vars if v in X_tr_noint.columns]
res_con_noint, Zc_noint = refit_with_constraints(long_tr_noint, thr_dum_tr_noint, X_tr_noint)
res_free_noint, Zf_noint, ref_thr_noint, thr_cols_noint = fit_stacked_logit(long_tr_noint, thr_dum_tr_noint, X_tr_noint, free_vars_noint)
res_null_noint, Zn_noint = refit_null(long_tr_noint, thr_dum_tr_noint)

LR_int, df_int, p_int = lr_test(res_free, res_free_noint)
lr_interaction_row = pd.DataFrame([{"LR_stat":LR_int,"df":df_int,"p_value":p_int}])

# ---------------- 3) 测试集预测 + 指标 ----------------
P_te = predict_gologit(res_free, X_te, thr_cols, free_vars, ref_thr)
y_pred_rank = P_te.argmax(axis=1) + classes_rank.min()
acc, mf1, kappa, mae, brier = ordered_metrics(y_te_rank.to_numpy(), y_pred_rank, P_te)

# 原始编码映射输出
y_pred_orig = pd.Series(y_pred_rank).map(inv_rank_map).to_numpy()
y_te_orig   = pd.Series(y_te_rank).map(inv_rank_map).to_numpy()
cm_orig = confusion_matrix(y_te_orig, y_pred_orig, labels=ordered_original)
report_txt = classification_report(y_te_orig, y_pred_orig, digits=3)

# ---------------- 4) 边际效应（总表 + age 在各 secu 水平下） ----------------
ame_df = average_marginal_effects_gologit(res_free, X_tr, thr_cols, free_vars, ref_thr, n_boot=N_BOOT_AME)
ame_age_secu = ame_age_within_secu(res_free, X_tr, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4)

# -------------- NEW: 附录 CSV（全量特征 × 各结局的 AME+95%CI） --------------
# 4.1 长表（每行：feature × class）
has_ci = {"AME_ci_low","AME_ci_high"}.issubset(set(ame_df.columns))
ame_long = ame_df.copy()
# 补上原始类别编码与可读标签
def _rank_to_label(rank_id):
    orig = inv_rank_map.get(rank_id, rank_id)
    # 给常用 2/1/3 命名；若不是这三类，退化为 "class-<orig>"
    default_names = {2:"No-injury", 1:"Minor-injury", 3:"Severe-injury"}
    return default_names.get(orig, f"class-{orig}")
ame_long["class_orig"]  = ame_long["class"].map(inv_rank_map)
ame_long["class_label"] = ame_long["class"].apply(_rank_to_label)
# 显著性（CI 不跨 0）
if has_ci:
    ame_long["is_sig"] = (ame_long["AME_ci_low"]>0)&(ame_long["AME_ci_high"]>0) | \
                         (ame_long["AME_ci_low"]<0)&(ame_long["AME_ci_high"]<0)
# 保存（原始概率单位）
csv_long_path = os.path.join(OUT_DIR, "appendix_AME_all_features_long.csv")
ame_long.to_csv(csv_long_path, index=False, encoding="utf-8-sig")

# 4.2 论文式“宽表”（每行：feature，每列：各结局，单元格=AME% [CI%]）
def _beautify(v):
    if v.startswith("age_x_secu_"): return v.replace("age_x_secu_", "Age × secu=") + " vs base"
    if v.startswith("secu_"):  return v.replace("secu_", "secu=") + " vs base"
    if v.startswith("obs_"):   return v.replace("obs_", "obs=") + " vs base"
    if v.startswith("plan_"):  return v.replace("plan_", "plan=") + " vs base"
    if v.startswith("lum_"):   return v.replace("lum_", "lum=") + " vs base"
    if v.startswith("sexe_"):  return v.replace("sexe_", "sexe=") + " vs base"
    if v.startswith("trajet_"):return v.replace("trajet_", "trajet=") + " vs base"
    if v=="age": return "Age (+1 SD)"
    if v=="age_secu": return "age_secu (numeric)"
    return v

wide_rows=[]
for v, g in ame_long.groupby("feature", sort=False):
    rec = {"Variables": _beautify(v)}
    for c in sorted(g["class"].unique()):
        gi = g[g["class"]==c].iloc[0]
        col = _rank_to_label(int(c))
        if has_ci:
            s = f"{gi['AME']*100:.2f} [{gi['AME_ci_low']*100:.2f}, {gi['AME_ci_high']*100:.2f}]"
            if gi["AME_ci_low"]*gi["AME_ci_high"]>0: s += " *"  # 星号=显著
        else:
            s = f"{gi['AME']*100:.2f}"
        rec[col] = s
    wide_rows.append(rec)
ame_wide = pd.DataFrame(wide_rows)
# 列顺序：Variables + 按 ordered_original 的标签顺序
cols = ["Variables"] + [_rank_to_label(rank_map[o]) for o in ordered_original]
cols = [c for c in cols if c in ame_wide.columns]
ame_wide = ame_wide.reindex(columns=cols)
csv_wide_path = os.path.join(OUT_DIR, "appendix_AME_all_features_wide.csv")
ame_wide.to_csv(csv_wide_path, index=False, encoding="utf-8-sig")

# ---------------- 5) 导出 Excel（保留你原先的所有工作表） ----------------
excel = os.path.join(OUT_DIR, "Gologit_review_ready.xlsx")
with pd.ExcelWriter(excel) as w:
    # A) 统计前提
    pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
    brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
    fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
    pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}]).to_excel(w, "A3_LR_test_partial_vs_constrained", index=False)
    extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr).to_excel(w, "A4_threshold_level_coefs", index=False)
    pd.DataFrame({"threshold_order":thr_order["threshold"],"intercept":thr_order["coef"],"is_monotone":[thr_monotone]*len(thr_order)}).to_excel(w,"A5_cutpoint_monotonicity",index=False)
    # B) 外推指标
    pd.DataFrame({"metric":["Accuracy","MacroF1","QuadraticKappa","OrdinalMAE","MulticlassBrier"],"value":[acc,mf1,kappa,mae,brier]}).to_excel(w,"B1_test_metrics",index=False)
    pd.DataFrame(cm_orig, index=[f"true_{c}" for c in ordered_original], columns=[f"pred_{c}" for c in ordered_original]).to_excel(w,"B2_confusion_matrix_orig")
    pd.DataFrame({"obs_id":np.arange(len(y_te_orig)),"y_test_orig":y_te_orig,"y_pred_orig":y_pred_orig}).to_excel(w,"B3_test_predictions_orig",index=False)
    ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)
    # C) 交互
    pd.DataFrame([{"LR_stat":LR_int,"df":df_int,"p_value":p_int}]).to_excel(w, "C1_LR_interaction_vs_base", index=False)
    (extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
      .query("term.str.startswith('age_x_secu_')", engine="python")
      .to_excel(w, "C2_interaction_coefs_by_threshold", index=False))
    ame_age_secu.to_excel(w, "C3_AME_age_by_secu", index=False)
    # D) 论文&附录表（可选）
    ame_wide.to_excel(w, "D1_AME_paper_table_all", index=False)

# ---------------- 6) 控制台摘要 ----------------
print("=== 统计前提（结构） ===")
print(f"Brant 全局: {brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}")
print("将放松的变量（含 age×secu 交互）：", list(free_vars))
print("\n约束 vs 放松 的拟合：\n", fit_table.to_string(index=False))
print("\n含交互 vs 不含交互（放松模型）：\n", pd.DataFrame([{'LR_stat':LR_int,'df':df_int,'p_value':p_int}]).to_string(index=False))
print("\n阈值截距是否单调递增？ ->", thr_monotone)
print("\n=== 外推（测试集，原始编码显示） ===")
print(f"Accuracy={acc:.3f}, Macro-F1={mf1:.3f}, Quadratic κ={kappa:.3f}, Ordinal MAE={mae:.3f}, Brier={brier:.4f}")
print("\nCSV（附录用，已保存所有特征 × 各结局的 AME+95%CI）：")
print("  - Long: ", csv_long_path)
print("  - Wide: ", csv_wide_path)
print("\nExcel 文件：", excel)


将放松非平行的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']
=== 统计前提（结构） ===
Brant 全局: 234.51, df=35, p=0
将放松的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']

约束 vs 放松 的拟合：
                   model          llf  df_model          aic          bic  pseudoR2_vsNull
Constrained_PO(stacked) -8040.191821      36.0 16154.383641 16447.617856         0.376476
  Partial_PO(Free vars) -7928.838286      53.0 15965.676573 16393.640021         0.385112

含交互 vs 不含交互（放松模型）：
  LR_stat  df  p_value
 7.60681   6 0.268347

阈值截距是否单调递增？ -> False

=== 外推（测试集，原始编码显示） ===
Accuracy=0.639, Macro-F1=0.647, Quadratic κ=0.646, Ordinal MAE=0.400, Brier=0.1475

CSV（附录用，已保存所有特征 × 各结局的 AME+95%CI）：
  - Lon

/tmp/ipython-input-3432689581.py:379: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
/tmp/ipython-input-3432689581.py:380: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
/tmp/ipython-input-3432689581.py:381: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
/tmp/ipython-input-3432689581.py:382: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the a

In [ ]:
# ============================================================
# Generalized Ordered Logit (Partial Proportional Odds) – Reviewer-ready
# + age × secu 交互；含交互 vs 不含交互 LR；AME(age|secu)
# + 导出“全量特征的 AME+95%CI”（长表&宽表）
# + NEW: 生成“论文截图风格”的 AME 表（Variables / Level-vs-Base / 三类概率 / 95%CI-Severe）
# ============================================================

import os, warnings, numpy as np, pandas as pd, statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
from scipy.special import expit
from scipy.stats import chi2
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- 配置 ----------------
FILE_PATH   = "//content/crash_data(final).csv"   # ← 改成你的CSV路径
TARGET      = "grav_balanced"
OUT_DIR     = "/content/gologit_outputs"; os.makedirs(OUT_DIR, exist_ok=True)
TEST_SIZE   = 0.2
RANDOM_STATE= 42
RARE_MIN    = 150
ALPHA_VAR   = 0.05
FORCE_FREE  = ["age"]
N_BOOT_AME  = 500     # ← >0 则为 AME 计算 95% CI
ORDINAL_ORDER = [2, 1, 3]   # 你的业务顺序：2(无伤)<1(轻伤)<3(重伤)

# ---------------- 工具函数 ----------------
def merge_rare(s, k=RARE_MIN):
    s = s.astype(str); vc = s.value_counts(dropna=True)
    return s.where(~s.isin(vc[vc<k].index), "__OTHER__")

def _build_rank_maps(y_series, ordinal_order):
    if ordinal_order is None: ordered = sorted(pd.unique(y_series))
    else: ordered = list(ordinal_order)
    rank_map = {v: i+1 for i, v in enumerate(ordered)}
    inv_rank_map = {r: v for v, r in rank_map.items()}
    return rank_map, inv_rank_map, ordered

def prep_data(path, ordinal_order=None):
    need = ["grav_balanced","secu","obs","plan","lum","sexe","trajet","age","age_secu"]
    df0  = pd.read_csv(path)
    use  = [c for c in need if c in df0.columns]
    if "grav_balanced" not in use: raise ValueError("缺少 grav_balanced")
    df = df0[use].dropna().copy()

    # 保留原始 age 的标准差（用于表格中 +1SD ≈ xx）
    age_raw = pd.to_numeric(df["age"], errors="coerce")
    age_raw_sd = float(np.nanstd(age_raw, ddof=0)) if "age" in df.columns else np.nan

    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype(int)
    rank_map, inv_rank_map, ordered_original = _build_rank_maps(df[TARGET], ordinal_order)
    y_orig = df[TARGET].copy()
    y_rank = df[TARGET].map(rank_map).astype(int)

    for c in ["secu","obs","plan","lum","sexe","trajet"]:
        if c in df.columns:
            df[c] = merge_rare(df[c], RARE_MIN)

    # 数值：age 标准化（使 +1SD = 1）
    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        sd = df["age"].std(ddof=0)
        df["age"] = (df["age"] - df["age"].mean())/(sd if sd!=0 else 1.0)
    if "age_secu" in df.columns:
        df["age_secu"] = pd.to_numeric(df["age_secu"], errors="coerce").fillna(0.0)

    # one-hot
    have_cats = [c for c in ["secu","obs","plan","lum","sexe","trajet"] if c in df.columns]
    X_cat = pd.get_dummies(df[have_cats], drop_first=True, dtype=float) if have_cats else pd.DataFrame(index=df.index)
    num_cols = ["age"] + (["age_secu"] if "age_secu" in df.columns else [])
    X_num = df[num_cols].astype(float) if num_cols else pd.DataFrame(index=df.index)

    # age × secu_* 交互
    secu_dummies = [c for c in X_cat.columns if c.startswith("secu_")]
    X_int = pd.DataFrame(index=df.index)
    for c in secu_dummies:
        X_int[f"age_x_{c}"] = X_num["age"] * X_cat[c]

    X = pd.concat([X_num, X_cat, X_int], axis=1)
    X = X.loc[:, X.std(ddof=0) > 0]
    return X, y_rank, y_orig, df, rank_map, inv_rank_map, ordered_original, secu_dummies, age_raw_sd

def brant_test_like(X, y_rank):
    levels = np.sort(y_rank.unique()); th = levels[:-1]
    betas, covs = [], []
    for k in th:
        yk = (y_rank > k).astype(int)
        res = sm.Logit(yk, sm.add_constant(X, has_constant='add')).fit(disp=False, maxiter=400)
        b = res.params.drop("const"); V = res.cov_params().loc[b.index, b.index]
        betas.append(b.values); covs.append(V.values)
    betas = np.stack(betas, axis=0)
    p = betas.shape[1]; K1 = betas.shape[0]

    diffs, Vblk = [], []
    for k in range(1,K1):
        diffs.append(betas[k]-betas[0]); Vblk.append(covs[k]+covs[0])
    d = np.concatenate(diffs)
    V = np.zeros(((K1-1)*p,(K1-1)*p))
    for i in range(K1-1): V[i*p:(i+1)*p, i*p:(i+1)*p] = Vblk[i]
    Vinv = np.linalg.pinv(V)
    from scipy.stats import chi2 as _chi2
    chi2_stat = float(d.T @ Vinv @ d); df_w = d.size; p_global = 1.0 - _chi2.cdf(chi2_stat, df_w)
    global_row = pd.DataFrame([{"scope":"global","chi2_stat":chi2_stat,"df":df_w,"p_value":p_global}])

    rows=[]
    for j,name in enumerate(X.columns):
        dj = np.array([betas[k,j]-betas[0,j] for k in range(1,K1)])
        Vj = np.diag([covs[k][j,j]+covs[0][j,j] for k in range(1,K1)])
        Vinvj = np.linalg.pinv(Vj)
        statj = float(dj.T @ Vinvj @ dj); dfj = dj.size; pj = 1.0 - _chi2.cdf(statj, dfj)
        rows.append({"feature":name,"chi2_stat":statj,"df":dfj,"p_value":pj})
    return global_row, pd.DataFrame(rows)

def stack_threshold_data(X, y_rank):
    classes = np.sort(y_rank.unique()); th = classes[:-1]
    n = len(y_rank); rep = len(th)
    long = pd.DataFrame({
        "obs_id": np.repeat(np.arange(n), rep),
        "thr":    np.tile(th, n),
        "y_bin":  (np.repeat(y_rank.to_numpy(), rep) > np.tile(th, n)).astype(int),
    })
    Xrep = np.repeat(X.to_numpy(), rep, axis=0)
    long = pd.concat([long, pd.DataFrame(Xrep, columns=X.columns)], axis=1)
    thr_dum = pd.get_dummies(long["thr"].astype(str), drop_first=False, dtype=int)
    thr_dum.columns = [f"thr_{c}" for c in thr_dum.columns]
    return long.reset_index(drop=True), thr_dum

def fit_stacked_logit(long_df, thr_dum, X, free_vars):
    Z = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    thr_cols = [c for c in thr_dum.columns]; ref_thr = thr_cols[0]
    for v in free_vars:
        for t in thr_cols[1:]:
            Z[f"{v}__x__{t}"] = long_df[v].to_numpy() * thr_dum[t].to_numpy()
    res = sm.Logit(long_df["y_bin"], Z).fit(disp=False, maxiter=400, cov_type="cluster",
                                            cov_kwds={"groups": long_df["obs_id"]})
    return res, Z, ref_thr, thr_cols

def refit_with_constraints(long_df, thr_dum, X):
    Zc = pd.concat([thr_dum.reset_index(drop=True), long_df[X.columns].reset_index(drop=True)], axis=1)
    resC = sm.Logit(long_df["y_bin"], Zc).fit(disp=False, maxiter=400, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resC, Zc

def refit_null(long_df, thr_dum):
    Zn = thr_dum.copy()
    resN = sm.Logit(long_df["y_bin"], Zn).fit(disp=False, maxiter=200, cov_type="cluster",
                                              cov_kwds={"groups": long_df["obs_id"]})
    return resN, Zn

def lr_test(res_full, res_small):
    LR = 2*(res_full.llf - res_small.llf)
    df = res_full.df_model - res_small.df_model
    p = 1.0 - chi2.cdf(LR, df if df>0 else 1)
    return LR, int(df), float(p)

def extract_threshold_coefs(res, Xcols, thr_cols, free_vars, ref_thr):
    coefs = res.params; cov = res.cov_params()
    rows=[]
    for t in thr_cols:
        b0 = coefs[t]; se0 = np.sqrt(cov.loc[t,t])
        rows.append({"threshold":t,"term":"intercept","coef":b0,"se":se0,
                     "ci_low":b0-1.96*se0,"ci_high":b0+1.96*se0,"OR":np.nan})
        for v in Xcols:
            if v in free_vars:
                if t==ref_thr:
                    b = coefs.get(v, 0.0); var = cov.loc[v,v]
                else:
                    dv = f"{v}__x__{t}"
                    b = coefs.get(v, 0.0)+coefs.get(dv, 0.0)
                    var = cov.loc[v,v]+cov.loc[dv,dv]+2*cov.loc[v,dv]
            else:
                b = coefs.get(v, 0.0); var = cov.loc[v,v]
            se = np.sqrt(var)
            rows.append({"threshold":t,"term":v,"coef":b,"se":se,
                         "ci_low":b-1.96*se,"ci_high":b+1.96*se,"OR":np.exp(b)})
    return pd.DataFrame(rows)

def predict_gologit(res, X, thr_cols, free_vars, ref_thr):
    coefs = res.params; Pgt=[]
    for t in thr_cols:
        eta = coefs[t]
        for v in X.columns:
            if v not in free_vars:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
        for v in free_vars:
            if t==ref_thr:
                eta += X[v].to_numpy() * coefs.get(v, 0.0)
            else:
                eta += X[v].to_numpy() * (coefs.get(v, 0.0)+coefs.get(f"{v}__x__{t}", 0.0))
        Pgt.append(expit(eta))
    Pgt = np.vstack(Pgt)
    p1 = 1.0 - Pgt[0]; p3 = Pgt[-1]; p2 = Pgt[0]-Pgt[-1]
    P = np.vstack([p1,p2,p3]).T
    P = np.clip(P, 1e-12, 1-1e-12); P = P / P.sum(axis=1, keepdims=True)
    return P

def ordered_metrics(y_true_rank, y_pred_rank, prob):
    acc = accuracy_score(y_true_rank, y_pred_rank)
    mf1 = f1_score(y_true_rank, y_pred_rank, average="macro")
    kappa = cohen_kappa_score(y_true_rank, y_pred_rank, weights="quadratic")
    mae = np.mean(np.abs(y_true_rank - y_pred_rank))
    K = prob.shape[1]; Y = np.eye(K)[(y_true_rank-1)]
    brier = np.mean(np.sum((prob - Y)**2, axis=1)) / K
    return acc, mf1, kappa, mae, brier

def average_marginal_effects_gologit(res, X, thr_cols, free_vars, ref_thr,
                                     h=1e-4, n_boot=0, seed=123, step_map=None):
    """
    计算 AME；支持对某些变量自定义“步长”：
    - 二元变量：默认把该列全设为 1.0
    - 连续变量：默认加 h（数值微分）
    - step_map: dict，如 {"age": 1.0, "age_x_secu_1": 1.0, ...}
                表示这些列用给定步长；其中 age_x_secu_* 只在对应 secu_*==1 的样本上加步长。
    """
    rng = np.random.default_rng(seed)
    if step_map is None:
        step_map = {}

    def probs(ex):
        return predict_gologit(res, ex, thr_cols, free_vars, ref_thr)

    baseP = probs(X); K = baseP.shape[1]
    out = []

    for v in X.columns:
        X1 = X.copy()

        # ---- 按变量决定步长 ----
        if v in step_map:
            step = step_map[v]
            if v.startswith("age_x_secu_"):
                # 只对该 secu 水平的样本加步长（等价于 age 在该水平 +1SD）
                secu_col = v.replace("age_x_", "")
                idx = X1[secu_col] == 1.0
                X1.loc[idx, v] = X1.loc[idx, v] + step
            else:
                X1[v] = X1[v] + step  # 连续变量用自定义步长
        else:
            # 默认策略：二元 -> 置为 1；连续 -> +h
            uniq = set(np.unique(X[v]))
            if uniq.issubset({0.0, 1.0}):
                X1[v] = 1.0
            else:
                X1[v] = X1[v] + h

        dP = probs(X1) - baseP
        ame = dP.mean(axis=0)
        for k in range(K):
            out.append({"feature": v, "class": k+1, "AME": float(ame[k])})

    ame_df = pd.DataFrame(out)

    # ---- 自助法置信区间（如需）----
    if n_boot > 0:
        boots = []
        for b in range(n_boot):
            idx = rng.choice(np.arange(len(X)), size=len(X), replace=True)
            Xb = X.iloc[idx].reset_index(drop=True)
            Pb = probs(Xb)
            for v in X.columns:
                X1 = Xb.copy()
                if v in step_map:
                    step = step_map[v]
                    if v.startswith("age_x_secu_"):
                        secu_col = v.replace("age_x_", "")
                        idb = X1[secu_col] == 1.0
                        X1.loc[idb, v] = X1.loc[idb, v] + step
                    else:
                        X1[v] = X1[v] + step
                else:
                    uniq = set(np.unique(Xb[v]))
                    if uniq.issubset({0.0, 1.0}):
                        X1[v] = 1.0
                    else:
                        X1[v] = X1[v] + h
                dP = probs(X1) - Pb
                ame = dP.mean(axis=0)
                for k in range(K):
                    boots.append({"feature": v, "class": k+1, "AME": float(ame[k]), "boot": b})

        boots = pd.DataFrame(boots)
        cis = []
        for (f, c), g in boots.groupby(["feature", "class"]):
            lo, hi = np.percentile(g["AME"], [2.5, 97.5])
            cis.append({"feature": f, "class": c, "AME_ci_low": lo, "AME_ci_high": hi})
        ame_df = ame_df.merge(pd.DataFrame(cis), on=["feature", "class"], how="left")

    return ame_df


def ame_age_within_secu(res, X, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4):
    baseP = predict_gologit(res, X, thr_cols, free_vars, ref_thr)
    out=[]
    Xb = X.copy(); Xb["age"] = Xb["age"] + h
    Pb = predict_gologit(res, Xb, thr_cols, free_vars, ref_thr)
    ame_base = (Pb - baseP).mean(axis=0)
    for k in range(baseP.shape[1]):
        out.append({"secu_level":"__BASE__", "class":k+1, "AME_age":float(ame_base[k])})
    for s in secu_dummies:
        X1 = X.copy(); X1["age"] = X1["age"] + h
        inter = f"age_x_{s}"
        if inter in X1.columns:
            idx = X1[s]==1.0
            X1.loc[idx, inter] = X1.loc[idx, inter] + h
        P1 = predict_gologit(res, X1, thr_cols, free_vars, ref_thr)
        ame = (P1 - baseP).mean(axis=0)
        for k in range(baseP.shape[1]):
            out.append({"secu_level":s.replace("secu_",""), "class":k+1, "AME_age":float(ame[k])})
    return pd.DataFrame(out)

# ---------------- 1) 读取 & Ordered Logit & Brant ----------------
X_all, y_all_rank, y_all_orig, df_all, rank_map, inv_rank_map, ordered_original, secu_dummies, age_raw_sd = prep_data(
    FILE_PATH, ordinal_order=ORDINAL_ORDER
)
classes_rank = np.sort(y_all_rank.unique())

X_tr, X_te, y_tr_rank, y_te_rank = train_test_split(
    X_all, y_all_rank, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all_rank
)

po = OrderedModel(endog=y_tr_rank, exog=X_tr, distr="logit")
po_res = po.fit(method="bfgs", maxiter=1000, disp=False)

brant_global, brant_by = brant_test_like(X_tr, y_tr_rank)

# ---------------- 放松变量选择（含 age × secu 交互） ----------------
violators = set(brant_by.loc[brant_by["p_value"]<ALPHA_VAR, "feature"].tolist())
for v in FORCE_FREE:
    if v in X_tr.columns: violators.add(v)
age_secu_inters = [c for c in X_tr.columns if c.startswith("age_x_secu_")]
for v in age_secu_inters: violators.add(v)
free_vars = sorted(list(violators)) or (["age"] if "age" in X_tr.columns else [])
print("将放松非平行的变量（含 age×secu 交互）：", free_vars)

# ---------------- 2) 叠栈logit：约束 vs 放松 ----------------
long_tr, thr_dum_tr = stack_threshold_data(X_tr, y_tr_rank)
res_con, Zc = refit_with_constraints(long_tr, thr_dum_tr, X_tr)
res_free, Zf, ref_thr, thr_cols = fit_stacked_logit(long_tr, thr_dum_tr, X_tr, free_vars)
res_null, Zn = refit_null(long_tr, thr_dum_tr)

LR, df_lr, p_lr = lr_test(res_free, res_con)
fit_table = pd.DataFrame([
    {"model":"Constrained_PO(stacked)","llf":res_con.llf,"df_model":res_con.df_model,"aic":res_con.aic,"bic":res_con.bic,
     "pseudoR2_vsNull": 1 - (res_con.llf / res_null.llf)},
    {"model":"Partial_PO(Free vars)","llf":res_free.llf,"df_model":res_free.df_model,"aic":res_free.aic,"bic":res_free.bic,
     "pseudoR2_vsNull": 1 - (res_free.llf / res_null.llf)},
])
lr_row = pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}])
coef_by_thr = extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
thr_order = coef_by_thr.query("term=='intercept'")[["threshold","coef"]]
thr_monotone = np.all(np.diff(thr_order["coef"].values) > 0)

# ---------------- 2.5) “含交互 vs 不含交互”的 LR 检验 ----------------
X_tr_noint = X_tr.drop(columns=[c for c in X_tr.columns if c.startswith("age_x_secu_")], errors="ignore")
long_tr_noint, thr_dum_tr_noint = stack_threshold_data(X_tr_noint, y_tr_rank)
free_vars_noint = [v for v in free_vars if v in X_tr_noint.columns]
res_con_noint, Zc_noint = refit_with_constraints(long_tr_noint, thr_dum_tr_noint, X_tr_noint)
res_free_noint, Zf_noint, ref_thr_noint, thr_cols_noint = fit_stacked_logit(long_tr_noint, thr_dum_tr_noint, X_tr_noint, free_vars_noint)
res_null_noint, Zn_noint = refit_null(long_tr_noint, thr_dum_tr_noint)

LR_int, df_int, p_int = lr_test(res_free, res_free_noint)
lr_interaction_row = pd.DataFrame([{"LR_stat":LR_int,"df":df_int,"p_value":p_int}])

# ---------------- 3) 测试集预测 + 指标 ----------------
P_te = predict_gologit(res_free, X_te, thr_cols, free_vars, ref_thr)
y_pred_rank = P_te.argmax(axis=1) + classes_rank.min()
acc, mf1, kappa, mae, brier = ordered_metrics(y_te_rank.to_numpy(), y_pred_rank, P_te)

# 原始编码映射输出
y_pred_orig = pd.Series(y_pred_rank).map(inv_rank_map).to_numpy()
y_te_orig   = pd.Series(y_te_rank).map(inv_rank_map).to_numpy()
cm_orig = confusion_matrix(y_te_orig, y_pred_orig, labels=ordered_original)
report_txt = classification_report(y_te_orig, y_pred_orig, digits=3)

# ---------------- 4) 边际效应（总表 + age 在各 secu 水平下） ----------------
# 指定 +1SD 的步长：age 用 +1.0；所有 age×secu_* 也用 +1.0（仅对对应 secu 水平样本加）
step_map = {"age": 1.0}
for s in [c for c in X_tr.columns if c.startswith("age_x_secu_")]:
    step_map[s] = 1.0

ame_df = average_marginal_effects_gologit(
    res_free, X_tr, thr_cols, free_vars, ref_thr,
    n_boot=N_BOOT_AME, step_map=step_map
)

ame_age_secu = ame_age_within_secu(res_free, X_tr, thr_cols, free_vars, ref_thr, secu_dummies, h=1e-4)

# -------------- 附录：全量 AME CSV（长表 & 宽表） --------------
has_ci = {"AME_ci_low","AME_ci_high"}.issubset(set(ame_df.columns))
ame_long = ame_df.copy()
def _rank_to_label(rank_id):
    orig = inv_rank_map.get(rank_id, rank_id)
    default_names = {2:"No-injury", 1:"Minor-injury", 3:"Severe-injury"}
    return default_names.get(orig, f"class-{orig}")
ame_long["class_orig"]  = ame_long["class"].map(inv_rank_map)
ame_long["class_label"] = ame_long["class"].apply(_rank_to_label)
if has_ci:
    ame_long["is_sig"] = (ame_long["AME_ci_low"]>0)&(ame_long["AME_ci_high"]>0) | \
                         (ame_long["AME_ci_low"]<0)&(ame_long["AME_ci_high"]<0)
csv_long_path = os.path.join(OUT_DIR, "appendix_AME_all_features_long.csv")
ame_long.to_csv(csv_long_path, index=False, encoding="utf-8-sig")

def _beautify(v):
    if v.startswith("age_x_secu_"): return v.replace("age_x_secu_", "Age × secu=") + " vs base"
    if v.startswith("secu_"):  return v.replace("secu_", "secu=") + " vs base"
    if v.startswith("obs_"):   return v.replace("obs_", "obs=") + " vs base"
    if v.startswith("plan_"):  return v.replace("plan_", "plan=") + " vs base"
    if v.startswith("lum_"):   return v.replace("lum_", "lum=") + " vs base"
    if v.startswith("sexe_"):  return v.replace("sexe_", "sexe=") + " vs base"
    if v.startswith("trajet_"):return v.replace("trajet_", "trajet=") + " vs base"
    if v=="age": return "Age (+1 SD)"
    if v=="age_secu": return "age_secu (numeric)"
    return v

wide_rows=[]
for v, g in ame_long.groupby("feature", sort=False):
    rec = {"Variables": _beautify(v)}
    for c in sorted(g["class"].unique()):
        gi = g[g["class"]==c].iloc[0]
        col = _rank_to_label(int(c))
        if has_ci:
            s = f"{gi['AME']*100:+.2f} [{gi['AME_ci_low']*100:+.2f}, {gi['AME_ci_high']*100:+.2f}]"
            if gi["AME_ci_low"]*gi["AME_ci_high"]>0: s += " *"
        else:
            s = f"{gi['AME']*100:+.2f}"
        rec[col] = s
    wide_rows.append(rec)
ame_wide = pd.DataFrame(wide_rows)
cols = ["Variables"] + [_rank_to_label(rank_map[o]) for o in ordered_original]
cols = [c for c in cols if c in ame_wide.columns]
ame_wide = ame_wide.reindex(columns=cols)
csv_wide_path = os.path.join(OUT_DIR, "appendix_AME_all_features_wide.csv")
ame_wide.to_csv(csv_wide_path, index=False, encoding="utf-8-sig")

# -------------- NEW：生成“论文截图风格”表（Table 8 风格） --------------
def build_table8_like(ame_long_df, age_sd_note):
    # 定义分组展示名（按你的变量名改这里的友好名称）
    pretty_group = {
        "age": "Age",
        "lum": "Lighting (lum)",
        "secu": "Safety equipment (secu)",
        "obs": "Obstacle (obs)",
        "plan": "Road-surface (plan)",
        "sexe": "Gender (sexe)",
        "trajet": "Trip purpose (trajet)",
    }
    # 把 feature 划分为 group 与 level
    feats = sorted(ame_long_df["feature"].unique(), key=lambda x: (x.split("_")[0], x))
    rows=[]
    # Age 行（+1SD ≈ xx）
    if "age" in ame_long_df["feature"].unique():
        g = ame_long_df[ame_long_df["feature"]=="age"]
        def cell(klass):
            gi = g[g["class_label"]==klass].iloc[0]
            if {"AME_ci_low","AME_ci_high"}.issubset(gi.index):
                s = f"{gi['AME']*100:+.2f}"
                return s
            else:
                return f"{gi['AME']*100:+.2f}"
        def ci_severe():
            gi = g[g["class_label"]=="Severe-injury"].iloc[0]
            if {"AME_ci_low","AME_ci_high"}.issubset(gi.index):
                s = f"[{gi['AME_ci_low']*100:+.2f}, {gi['AME_ci_high']*100:+.2f}]"
                if gi["AME_ci_low"]*gi["AME_ci_high"]>0: s += " *"
                return s
            return ""
        rows.append({
            "Variables": "Age",
            "Level-vs-Base": f"+1 SD ≈ {age_sd_note}",
            "Minor-injury": cell("Minor-injury"),
            "No-injury":    cell("No-injury"),
            "Severe-injury":cell("Severe-injury"),
            "95%CI-Severe": ci_severe()
        })

    # 其它分组：从 one-hot 名字里提取 group & level
    for group in ["lum","secu","obs","plan","sexe","trajet"]:
        prefix = group + "_"
        sub = ame_long_df[ame_long_df["feature"].str.startswith(prefix)]
        if sub.empty: continue
        for feat in sorted(sub["feature"].unique()):
            g = sub[sub["feature"]==feat]
            level_name = feat.replace(prefix, "")
            def cell(klass):
                gi = g[g["class_label"]==klass].iloc[0]
                s = f"{gi['AME']*100:+.2f}"
                return s
            def ci_severe():
                gi = g[g["class_label"]=="Severe-injury"].iloc[0]
                s = f"[{gi['AME_ci_low']*100:+.2f}, {gi['AME_ci_high']*100:+.2f}]"
                if gi["AME_ci_low"]*gi["AME_ci_high"]>0: s += " *"
                return s
            rows.append({
                "Variables": pretty_group.get(group, group),
                "Level-vs-Base": f"{level_name} vs base",
                "Minor-injury": cell("Minor-injury"),
                "No-injury":    cell("No-injury"),
                "Severe-injury":cell("Severe-injury"),
                "95%CI-Severe": ci_severe()
            })
    return pd.DataFrame(rows)

age_sd_note = f"{age_raw_sd:.1f}y" if not np.isnan(age_raw_sd) else "1 SD"
table8_like = build_table8_like(ame_long, age_sd_note)
csv_table8 = os.path.join(OUT_DIR, "table8_like.csv")
table8_like.to_csv(csv_table8, index=False, encoding="utf-8-sig")

# ---------------- 5) 导出 Excel（含“论文截图风格”表） ----------------
excel = os.path.join(OUT_DIR, "Gologit_review_ready.xlsx")
with pd.ExcelWriter(excel) as w:
    # A) 统计前提
    pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
    brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
    fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
    pd.DataFrame([{"LR_stat":LR,"df":df_lr,"p_value":p_lr}]).to_excel(w, "A3_LR_test_partial_vs_constrained", index=False)
    extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr).to_excel(w, "A4_threshold_level_coefs", index=False)
    pd.DataFrame({"threshold_order":thr_order["threshold"],"intercept":thr_order["coef"],"is_monotone":[thr_monotone]*len(thr_order)}).to_excel(w,"A5_cutpoint_monotonicity",index=False)
    # B) 外推指标
    pd.DataFrame({"metric":["Accuracy","MacroF1","QuadraticKappa","OrdinalMAE","MulticlassBrier"],"value":[acc,mf1,kappa,mae,brier]}).to_excel(w,"B1_test_metrics",index=False)
    pd.DataFrame(cm_orig, index=[f"true_{c}" for c in ordered_original], columns=[f"pred_{c}" for c in ordered_original]).to_excel(w,"B2_confusion_matrix_orig")
    pd.DataFrame({"obs_id":np.arange(len(y_te_orig)),"y_test_orig":y_te_orig,"y_pred_orig":y_pred_orig}).to_excel(w,"B3_test_predictions_orig",index=False)
    ame_df.to_excel(w,"B4_marginal_effects_AME",index=False)
    # C) 交互
    pd.DataFrame([{"LR_stat":LR_int,"df":df_int,"p_value":p_int}]).to_excel(w, "C1_LR_interaction_vs_base", index=False)
    (extract_threshold_coefs(res_free, X_tr.columns, thr_cols, free_vars, ref_thr)
      .query("term.str.startswith('age_x_secu_')", engine="python")
      .to_excel(w, "C2_interaction_coefs_by_threshold", index=False))
    ame_age_secu.to_excel(w, "C3_AME_age_by_secu", index=False)
    # D) 论文&附录表
    ame_wide.to_excel(w, "D1_AME_paper_table_all", index=False)
    table8_like.to_excel(w, "TBL_AME_like", index=False)

# ---------------- 6) 控制台摘要 ----------------
print("=== 统计前提（结构） ===")
print(f"Brant 全局: {brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}")
print("将放松的变量（含 age×secu 交互）：", list(free_vars))
print("\n约束 vs 放松 的拟合：\n", fit_table.to_string(index=False))
print("\n含交互 vs 不含交互（放松模型）： χ²={:.2f}, df={}, p={:.3g}".format(LR_int, df_int, p_int))
print("\n阈值截距是否单调递增？ ->", thr_monotone)
print("\n=== 外推（测试集，原始编码显示） ===")
print(f"Accuracy={acc:.3f}, Macro-F1={mf1:.3f}, Quadratic κ={kappa:.3f}, Ordinal MAE={mae:.3f}, Brier={brier:.4f}")
print("\nCSV（附录用）:")
print("  - Long (全量): ", csv_long_path)
print("  - Wide (全量): ", csv_wide_path)
print("  - Table8-like: ", csv_table8)
print("\nExcel 文件：", excel)


将放松非平行的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']
=== 统计前提（结构） ===
Brant 全局: 234.51, df=35, p=0
将放松的变量（含 age×secu 交互）： ['age', 'age_x_secu_1', 'age_x_secu_2', 'age_x_secu_3', 'lum_2', 'lum_3', 'lum_4', 'lum_5', 'obs_1.0', 'obs_14.0', 'obs_16.0', 'plan_2', 'plan_3', 'plan_4', 'secu_1', 'sexe_2', 'trajet_5.0']

约束 vs 放松 的拟合：
                   model          llf  df_model          aic          bic  pseudoR2_vsNull
Constrained_PO(stacked) -8040.191821      36.0 16154.383641 16447.617856         0.376476
  Partial_PO(Free vars) -7928.838286      53.0 15965.676573 16393.640021         0.385112

含交互 vs 不含交互（放松模型）： χ²=7.61, df=6, p=0.268

阈值截距是否单调递增？ -> False

=== 外推（测试集，原始编码显示） ===
Accuracy=0.639, Macro-F1=0.647, Quadratic κ=0.646, Ordinal MAE=0.400, Brier=0.1475

CSV（附录用）:
  - Long (全量):  /content/gologit_outputs/appendix_AME_

/tmp/ipython-input-2700437535.py:514: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame({"note":[f"Brant global: chi2={brant_global['chi2_stat'].iloc[0]:.2f}, df={int(brant_global['df'].iloc[0])}, p={brant_global['p_value'].iloc[0]:.3g}"]}).to_excel(w,"A0_brant_global",index=False)
/tmp/ipython-input-2700437535.py:515: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  brant_by.sort_values("p_value").to_excel(w, "A1_brant_by_feature", index=False)
/tmp/ipython-input-2700437535.py:516: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  fit_table.to_excel(w, "A2_fit_PO_vs_Partial", index=False)
/tmp/ipython-input-2700437535.py:517: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the a